In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np

print("FastReasonSeg Implementation Workspace")
print("-" * 40)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Select T4 GPU in Runtime settings.")

print("\nWorkspace ready.")

In [ ]:
# ============================================================
# TASK 6: SPATIAL REASONING ENGINE + INDEPENDENT TESTS
# ============================================================

import math
import json

def center(box):
    """Calculate the center of [x1, y1, x2, y2]."""
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def area(box):
    """Calculate bounding-box area."""
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)


def spatial_relation(obj_a, obj_b):
    """
    Return geometric relationships between two objects.

    Coordinates use image convention:
    x increases to the right; y increases downward.
    """
    a = obj_a["bbox"]
    b = obj_b["bbox"]

    ax, ay = center(a)
    bx, by = center(b)

    relations = {
        "left_of": ax < bx,
        "right_of": ax > bx,
        "above": ay < by,
        "below": ay > by,
        "nearer_in_image": math.dist((ax, ay), (bx, by)) < 100,
    }

    # Bounding-box intersection
    ix1 = max(a[0], b[0])
    iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2])
    iy2 = min(a[3], b[3])

    intersection = (
        max(0, ix2 - ix1) * max(0, iy2 - iy1)
    )

    relations["overlapping"] = intersection > 0

    # Whether A's bounding box is entirely inside B's
    relations["inside"] = (
        a[0] >= b[0]
        and a[1] >= b[1]
        and a[2] <= b[2]
        and a[3] <= b[3]
    )

    relations["larger_bbox"] = area(a) > area(b)

    return relations


# ------------------------------------------------------------
# INDEPENDENT SYNTHETIC TEST CASES
# Expected results are specified manually.
# ------------------------------------------------------------

test_cases = [
    {
        "name": "Left relationship",
        "a": {"bbox": [0, 0, 20, 20]},
        "b": {"bbox": [100, 0, 120, 20]},
        "relation": "left_of",
        "expected": True,
    },
    {
        "name": "Right relationship",
        "a": {"bbox": [100, 0, 120, 20]},
        "b": {"bbox": [0, 0, 20, 20]},
        "relation": "right_of",
        "expected": True,
    },
    {
        "name": "Above relationship",
        "a": {"bbox": [0, 0, 20, 20]},
        "b": {"bbox": [0, 100, 20, 120]},
        "relation": "above",
        "expected": True,
    },
    {
        "name": "Below relationship",
        "a": {"bbox": [0, 100, 20, 120]},
        "b": {"bbox": [0, 0, 20, 20]},
        "relation": "below",
        "expected": True,
    },
    {
        "name": "Overlap detection",
        "a": {"bbox": [0, 0, 30, 30]},
        "b": {"bbox": [20, 20, 50, 50]},
        "relation": "overlapping",
        "expected": True,
    },
    {
        "name": "No overlap",
        "a": {"bbox": [0, 0, 10, 10]},
        "b": {"bbox": [20, 20, 30, 30]},
        "relation": "overlapping",
        "expected": False,
    },
    {
        "name": "Inside relationship",
        "a": {"bbox": [10, 10, 20, 20]},
        "b": {"bbox": [0, 0, 30, 30]},
        "relation": "inside",
        "expected": True,
    },
    {
        "name": "Larger bounding box",
        "a": {"bbox": [0, 0, 100, 100]},
        "b": {"bbox": [0, 0, 10, 10]},
        "relation": "larger_bbox",
        "expected": True,
    },
]

test_results = []

for test in test_cases:
    result = spatial_relation(test["a"], test["b"])
    actual = result[test["relation"]]
    passed = actual == test["expected"]

    test_results.append({
        "test": test["name"],
        "expected": test["expected"],
        "actual": actual,
        "passed": passed,
    })

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{test['name']} | "
        f"Expected={test['expected']} | Actual={actual}"
    )

passed_count = sum(r["passed"] for r in test_results)

print("\n========== TEST SUMMARY ==========")
print("Total tests:", len(test_results))
print("Passed:", passed_count)
print("Failed:", len(test_results) - passed_count)
print(f"Test pass rate: {passed_count / len(test_results) * 100:.1f}%")

# Save test results
with open("/content/spatial_reasoning_tests.json", "w") as f:
    json.dump(test_results, f, indent=2)

print("\nSaved: /content/spatial_reasoning_tests.json")

assert passed_count == len(test_results), (
    "Some tests failed. Inspect the failed cases."
)

print("Spatial reasoning unit tests completed successfully.")

In [ ]:
# TASK 7: CHECK VIDEO AND PROJECT FILES

import os

print("Checking current Colab files...\n")

folders = ["/content", "/content/drive/MyDrive"]

for folder in folders:
    if os.path.exists(folder):
        print(f"\nFolder: {folder}")
        try:
            for name in os.listdir(folder)[:30]:
                path = os.path.join(folder, name)
                print("  ", name)
        except Exception as e:
            print("Could not list folder:", e)
    else:
        print(f"\nFolder not available: {folder}")

print("\nSearching for video files in /content...")

video_extensions = (".mp4", ".avi", ".mov", ".mkv")

for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in [".cache", "__pycache__"]]
    for name in files:
        if name.lower().endswith(video_extensions):
            print(os.path.join(root, name))

print("\nFile check completed.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

root = "/content/drive/MyDrive"

keywords = [
    "FastReasonSeg",
    "digital_twin.json",
    "segmented_dog",
    "01_dog.mp4",
    "video_tracking_diagnostics"
]

matches = []

for folder, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in [".Trash", ".cache"]]

    for filename in files:
        if any(k.lower() in filename.lower() for k in keywords):
            path = os.path.join(folder, filename)
            matches.append(path)
            print(path)

print("\nTotal matching files:", len(matches))

In [ ]:
import json
import os

notebook_path = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "FastReasonSeg_Project.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Notebook:", os.path.basename(notebook_path))
print("Total cells:", len(notebook["cells"]))

print("\n===== CODE CELL HEADINGS =====")

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "markdown":
        text = "".join(cell.get("source", []))
        headings = [
            line.strip()
            for line in text.splitlines()
            if line.strip().startswith("#")
        ]
        for heading in headings[:3]:
            print(f"Cell {i}: {heading[:150]}")

print("\nNotebook inspection completed.")

In [ ]:
import json
import re

path = "/content/drive/MyDrive/Colab Notebooks/FastReasonSeg_Project.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("PROJECT IMPLEMENTATION MAP\n")

keywords = [
    "owlv2", "grounding dino", "sam2", "depth anything",
    "digital twin", "reason", "video", "tracking",
    "dataset", "qwen", "teacher", "sft", "grpo",
    "reward", "student", "benchmark", "evaluation",
    "train", "save", "iou", "dice"
]

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))
    matches = [
        word for word in keywords
        if re.search(re.escape(word), source, re.IGNORECASE)
    ]

    if matches:
        preview = " ".join(source.strip().split())[:180]
        print(f"Cell {i} | Keywords: {', '.join(matches)}")
        print(" ", preview, "\n")

print("Inspection complete.")

In [ ]:
import os
import urllib.request

video_url = "https://dl.fbaipublicfiles.com/segment_anything_2/assets/01_dog.mp4"
video_path = "/content/01_dog.mp4"

if not os.path.exists(video_path):
    print("Downloading test video...")
    urllib.request.urlretrieve(video_url, video_path)

print("Video exists:", os.path.exists(video_path))
print("Video size:", round(os.path.getsize(video_path) / (1024 * 1024), 2), "MB")
print("Video path:", video_path)

In [ ]:
import os
import urllib.request

video_urls = [
    "https://raw.githubusercontent.com/facebookresearch/sam2/main/demo/data/gallery/01_dog.mp4",
    "https://github.com/facebookresearch/sam2/raw/main/demo/data/gallery/01_dog.mp4",
]

video_path = "/content/01_dog.mp4"
downloaded = False

for url in video_urls:
    try:
        request = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(request, timeout=30) as response:
            with open(video_path, "wb") as f:
                f.write(response.read())

        if os.path.getsize(video_path) > 100_000:
            downloaded = True
            break

    except Exception as e:
        print("This source failed:", type(e).__name__, str(e)[:150])

if downloaded:
    print("SUCCESS: Video downloaded")
    print("Size:", round(os.path.getsize(video_path) / (1024 * 1024), 2), "MB")

    import cv2
    cap = cv2.VideoCapture(video_path)
    print("Video opened:", cap.isOpened())
    print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    print("Resolution:",
          int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "x",
          int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    cap.release()
else:
    print("Both download sources failed.")
    print("Use the upload option below instead.")

In [ ]:
import torch
import os

assert os.path.exists("/content/01_dog.mp4"), "Video file missing!"
assert torch.cuda.is_available(), "GPU is not enabled!"

print("GPU:", torch.cuda.get_device_name(0))

from sam2.sam2_video_predictor import SAM2VideoPredictor

print("Loading SAM2 video predictor...")

video_predictor = SAM2VideoPredictor.from_pretrained(
    "facebook/sam2.1-hiera-small"
)

print("SAM2 video predictor loaded!")

In [ ]:
%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd /content/sam2
!pip install -e .

In [ ]:
import os
import sys
import importlib.util
import subprocess

print("Python:", sys.executable)
print("Working directory:", os.getcwd())

repo = "/content/sam2"
print("SAM2 repository exists:", os.path.isdir(repo))
print("Video predictor file exists:",
      os.path.isfile(repo + "/sam2/sam2_video_predictor.py"))

print("\nImport resolution:")
spec = importlib.util.find_spec("sam2")
print("sam2 spec:", spec)
print("sam2 search locations:",
      list(spec.submodule_search_locations or []) if spec else None)

print("\nInstalled package:")
subprocess.run([sys.executable, "-m", "pip", "show", "sam2"])

print("\nRepository files:")
if os.path.isdir(repo):
    print(os.listdir(repo)[:20])
    package_dir = repo + "/sam2"
    if os.path.isdir(package_dir):
        print("Package files:", os.listdir(package_dir)[:20])

In [ ]:
%cd /content/sam2

!python -m pip install -e . --no-build-isolation

In [ ]:
import os
import sys

os.chdir("/tmp")

# Remove any stale namespace-package entry
sys.modules.pop("sam2", None)

import sam2
print("SAM2 package:", sam2)
print("Package file:", getattr(sam2, "__file__", None))
print("Package paths:", list(sam2.__path__))

from sam2.sam2_video_predictor import SAM2VideoPredictor

print("SUCCESS: SAM2 video predictor imported!")

In [ ]:
import os
import sys
import importlib.util
import importlib.metadata

os.chdir("/tmp")
sys.path.insert(0, "/content/sam2")

print("SAM-2 distribution:",
      importlib.metadata.version("SAM-2"))

print("\nPython search paths:")
for p in sys.path[:8]:
    print(p)

print("\nSAM2 package spec:")
print(importlib.util.find_spec("sam2"))

print("\nVideo predictor file:")
print(os.path.isfile("/content/sam2/sam2/sam2_video_predictor.py"))

print("\nPackage initializer:")
print(open("/content/sam2/sam2/__init__.py").read()[:500])

In [ ]:
import os
import sys
import importlib

os.chdir("/tmp")

# Put the repository root first
sys.path.insert(0, "/content/sam2")

# Clear stale SAM2 modules from the current Python session
for name in list(sys.modules):
    if name == "sam2" or name.startswith("sam2."):
        del sys.modules[name]

importlib.invalidate_caches()

from sam2.sam2_video_predictor import SAM2VideoPredictor

print("SUCCESS: SAM2 video predictor imported!")

In [ ]:
import torch
from sam2.sam2_video_predictor import SAM2VideoPredictor

print("GPU:", torch.cuda.get_device_name(0))
print("Loading SAM2.1 Small video model...")

video_predictor = SAM2VideoPredictor.from_pretrained(
    "facebook/sam2.1-hiera-small"
)

print("SUCCESS: Video model loaded!")

In [ ]:
import decord

print("Decord version:", decord.__version__)

video_path = "/content/01_dog.mp4"

inference_state = video_predictor.init_state(
    video_path=video_path
)

print("SUCCESS: Video initialized!")

In [ ]:
%pip install decord

In [ ]:
import decord
import os

print("Decord version:", decord.__version__)

video_path = "/content/01_dog.mp4"
assert os.path.exists(video_path), "Video file is missing!"

inference_state = video_predictor.init_state(
    video_path=video_path
)

print("SUCCESS: Video initialized!")


In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = "/content/01_dog.mp4"

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

assert ret, "Could not read the first frame"

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 7))
plt.imshow(frame_rgb)
plt.title("First frame — select a point on the dog")
plt.axis("on")
plt.show()

print("Frame dimensions (height, width):", frame_rgb.shape[:2])

In [ ]:
from IPython.display import display, Image
import cv2

cap = cv2.VideoCapture("/content/01_dog.mp4")
ret, frame = cap.read()
cap.release()

assert ret, "Could not read frame"

cv2.imwrite("/content/first_frame.jpg", frame)
display(Image(filename="/content/first_frame.jpg", width=800))

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Display the first frame
plt.figure(figsize=(12, 7))
plt.imshow(frame_rgb)
plt.title("First frame: identify a point on the dog")
plt.axis("on")
plt.show()

In [ ]:
# Change these coordinates to a point INSIDE the dog
x, y = 650, 450

points = np.array([[x, y]], dtype=np.float32)
labels = np.array([1], dtype=np.int32)  # 1 = foreground

with torch.inference_mode():
    frame_idx, object_ids, masks = video_predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=0,
        obj_id=1,
        points=points,
        labels=labels,
    )

print("Frame:", frame_idx)
print("Object IDs:", object_ids)
print("Mask shape:", masks.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# Extract the initial mask
initial_mask = masks[0, 0].detach().float().cpu().numpy() > 0

print("Mask shape:", initial_mask.shape)
print("Foreground pixels:", int(initial_mask.sum()))
print("Mask coverage:", round(initial_mask.mean() * 100, 2), "%")

# Display original frame and mask overlay
plt.figure(figsize=(14, 7))
plt.imshow(frame_rgb)
plt.imshow(initial_mask, alpha=0.5, cmap="spring")
plt.title("SAM2 Initial Segmentation")
plt.axis("off")
plt.show()

In [ ]:
import torch
import numpy as np

video_results = {}

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        for frame_idx, object_ids, masks in video_predictor.propagate_in_video(
            inference_state
        ):
            frame_masks = {}

            for i, obj_id in enumerate(object_ids):
                mask = (
                    masks[i, 0].detach().float().cpu().numpy() > 0
                )
                frame_masks[int(obj_id)] = mask

            video_results[int(frame_idx)] = frame_masks

print("Video propagation finished!")
print("Frames processed:", len(video_results))
print("Total video frames: 289")
print("Coverage:", round(len(video_results) / 289 * 100, 2), "%")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

records = []

for frame_idx in sorted(video_results.keys()):
    frame_masks = video_results[frame_idx]
    mask = frame_masks.get(1)

    if mask is None:
        records.append({
            "frame": frame_idx,
            "mask_pixels": 0,
            "mask_area_percent": 0.0,
            "centroid_x": np.nan,
            "centroid_y": np.nan,
            "status": "Missing mask"
        })
        continue

    mask = np.asarray(mask, dtype=bool)
    ys, xs = np.where(mask)
    pixels = int(mask.sum())

    records.append({
        "frame": frame_idx,
        "mask_pixels": pixels,
        "mask_area_percent": round(pixels / mask.size * 100, 4),
        "centroid_x": round(float(xs.mean()), 2) if pixels else np.nan,
        "centroid_y": round(float(ys.mean()), 2) if pixels else np.nan,
        "status": "Valid mask" if pixels else "Empty mask"
    })

tracking_df = pd.DataFrame(records)

print("Total frames:", len(tracking_df))
print("Missing masks:", (tracking_df["status"] == "Missing mask").sum())
print("Empty masks:", (tracking_df["status"] == "Empty mask").sum())
print("Valid masks:", (tracking_df["status"] == "Valid mask").sum())
print("\nMask area statistics:")
print(tracking_df["mask_area_percent"].describe().round(4))

display(tracking_df.head(10))

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(
    tracking_df["frame"],
    tracking_df["mask_area_percent"],
    linewidth=1
)
plt.xlabel("Frame")
plt.ylabel("Mask area (% of frame)")
plt.title("SAM2 Dog Tracking — Segmented Area Over Time")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/tracking_quality_plot.png", dpi=150)
plt.show()

print("Plot saved: /content/tracking_quality_plot.png")

In [ ]:
from IPython.display import display, Image

display(Image(filename="/content/tracking_quality_plot.png"))

print("Total frames:", len(tracking_df))
print("Missing masks:", (tracking_df["status"] == "Missing mask").sum())
print("Empty masks:", (tracking_df["status"] == "Empty mask").sum())
print("Valid masks:", (tracking_df["status"] == "Valid mask").sum())

In [ ]:
# Identify frames where SAM2 returned an empty mask
empty_frames = tracking_df.loc[
    tracking_df["status"] == "Empty mask", "frame"
].tolist()

print("Number of empty frames:", len(empty_frames))
print("Empty frame indices:", empty_frames)

# Save the list
import json

with open("/content/empty_frames.json", "w") as f:
    json.dump(empty_frames, f, indent=2)

print("Saved: /content/empty_frames.json")

In [ ]:
import os
import json

output_dir = "/content/FastReasonSeg_Results"
os.makedirs(output_dir, exist_ok=True)

# Export per-frame metrics
tracking_df.to_csv(
    os.path.join(output_dir, "tracking_metrics.csv"),
    index=False
)

# Calculate summary statistics
summary = {
    "video": "/content/01_dog.mp4",
    "total_frames": int(len(tracking_df)),
    "valid_masks": int((tracking_df["status"] == "Valid mask").sum()),
    "empty_masks": int((tracking_df["status"] == "Empty mask").sum()),
    "missing_masks": int((tracking_df["status"] == "Missing mask").sum()),
    "mask_availability_percent": round(
        (tracking_df["status"] == "Valid mask").mean() * 100, 2
    ),
    "note": (
        "Mask availability is not segmentation accuracy. "
        "Ground-truth masks are required to measure segmentation accuracy."
    )
}

with open(os.path.join(output_dir, "evaluation_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

# Copy the tracking plot
import shutil
shutil.copy(
    "/content/tracking_quality_plot.png",
    os.path.join(output_dir, "tracking_quality_plot.png")
)

print(json.dumps(summary, indent=2))
print("\nExported results to:", output_dir)

In [ ]:
import cv2
import numpy as np
import os

video_path = "/content/01_dog.mp4"
output_dir = "/content/FastReasonSeg_Results"
output_video = os.path.join(output_dir, "segmented_dog.mp4")

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_masks = video_results.get(frame_idx, {})
    mask = frame_masks.get(1)

    if mask is not None:
        mask = np.asarray(mask, dtype=bool)

        # Overlay only nonempty masks
        if mask.any():
            overlay = frame.copy()
            overlay[mask] = (
                0.3 * overlay[mask] + 0.7 * np.array([0, 255, 0])
            ).astype(np.uint8)

            frame = cv2.addWeighted(frame, 0.55, overlay, 0.45, 0)

    # Add frame number
    cv2.putText(
        frame, f"Frame: {frame_idx}",
        (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
        1, (255, 255, 255), 2
    )

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

print("Video created:", output_video)
print("Frames written:", frame_idx)
print("FPS:", fps)
print("Resolution:", width, "x", height)
print("File size (MB):", round(os.path.getsize(output_video) / (1024**2), 2))

In [ ]:
import os
import zipfile

output_dir = "/content/FastReasonSeg_Results"
zip_path = "/content/FastReasonSeg_Prototype_Results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_dir):
        for filename in files:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, output_dir)
            zipf.write(filepath, arcname)

print("ZIP created:", zip_path)
print("ZIP size (MB):", round(os.path.getsize(zip_path) / (1024**2), 2))
print("\nFiles included:")
with zipfile.ZipFile(zip_path, "r") as zipf:
    for name in zipf.namelist():
        print("-", name)

In [ ]:
import os
import json
import zipfile

output_dir = "/content/FastReasonSeg_Results"

# Save empty-frame report
empty_frames = tracking_df.loc[
    tracking_df["status"] == "Empty mask", "frame"
].astype(int).tolist()

empty_path = os.path.join(output_dir, "empty_frames.json")

with open(empty_path, "w") as f:
    json.dump(empty_frames, f, indent=2)

# Rebuild ZIP with all files
zip_path = "/content/FastReasonSeg_Prototype_Results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_dir):
        for filename in files:
            path = os.path.join(root, filename)
            zipf.write(path, os.path.relpath(path, output_dir))

# Verify package
with zipfile.ZipFile(zip_path, "r") as zipf:
    print("ZIP verified:", zipf.testzip() is None)
    print("\nFiles included:")
    for name in zipf.namelist():
        print("-", name)

print("\nFinal ZIP size:", round(os.path.getsize(zip_path) / (1024**2), 2), "MB")

In [ ]:
from google.colab import files
files.download("/content/FastReasonSeg_Prototype_Results.zip")

In [ ]:
import numpy as np
import pandas as pd
import json
import os

iou_scores = []

for frame_idx in range(1, len(video_results)):
    previous = video_results.get(frame_idx - 1, {}).get(1)
    current = video_results.get(frame_idx, {}).get(1)

    if previous is None or current is None:
        continue

    previous = np.asarray(previous, dtype=bool)
    current = np.asarray(current, dtype=bool)

    # Skip empty masks
    if not previous.any() or not current.any():
        continue

    intersection = np.logical_and(previous, current).sum()
    union = np.logical_or(previous, current).sum()

    if union > 0:
        iou_scores.append({
            "frame": frame_idx,
            "consecutive_frame_iou": intersection / union
        })

iou_df = pd.DataFrame(iou_scores)

if len(iou_df):
    print("Valid consecutive-frame comparisons:", len(iou_df))
    print("Mean temporal IoU:", round(iou_df["consecutive_frame_iou"].mean(), 4))
    print("Median temporal IoU:", round(iou_df["consecutive_frame_iou"].median(), 4))
    print("Minimum temporal IoU:", round(iou_df["consecutive_frame_iou"].min(), 4))

    iou_df.to_csv(
        "/content/FastReasonSeg_Results/temporal_stability.csv",
        index=False
    )
    print("Saved temporal_stability.csv")
else:
    print("No valid consecutive-frame comparisons.")

In [ ]:
import os
import json
import zipfile
import numpy as np

output_dir = "/content/FastReasonSeg_Results"
os.makedirs(output_dir, exist_ok=True)

# Load existing evaluation summary if available
summary_path = os.path.join(output_dir, "evaluation_summary.json")

if os.path.exists(summary_path):
    with open(summary_path, "r") as f:
        summary = json.load(f)
else:
    summary = {}

# Add temporal stability results
summary["temporal_stability"] = {
    "valid_consecutive_frame_comparisons": int(len(iou_df)),
    "mean_temporal_iou": round(float(iou_df["consecutive_frame_iou"].mean()), 4),
    "median_temporal_iou": round(float(iou_df["consecutive_frame_iou"].median()), 4),
    "minimum_temporal_iou": round(float(iou_df["consecutive_frame_iou"].min()), 4),
    "metric_note": (
        "Consecutive-frame mask overlap; not ground-truth segmentation accuracy."
    )
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

# Rebuild ZIP with all current outputs
zip_path = "/content/FastReasonSeg_Prototype_Results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk(output_dir):
        for filename in files_in_dir:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, output_dir)
            zipf.write(filepath, arcname)

print(json.dumps(summary, indent=2))
print("\nUpdated ZIP:", zip_path)

with zipfile.ZipFile(zip_path, "r") as zipf:
    print("\nZIP verification:", "PASS" if zipf.testzip() is None else "FAIL")
    print("Files:", zipf.namelist())

In [ ]:
print(
    iou_df.sort_values("consecutive_frame_iou")
    .head(5)
    .to_string(index=False)
)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

video_path = "/content/01_dog.mp4"
frames_to_check = [74, 75, 76, 77, 78, 101, 102, 103]

cap = cv2.VideoCapture(video_path)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for ax, idx in zip(axes, frames_to_check):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()

    if not ret:
        ax.set_title(f"Frame {idx}: unavailable")
        ax.axis("off")
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mask = video_results.get(idx, {}).get(1)

    ax.imshow(frame)

    if mask is not None and np.asarray(mask).any():
        overlay = np.zeros_like(frame)
        overlay[:, :, 1] = 255
        ax.imshow(np.where(
            np.asarray(mask, dtype=bool)[:, :, None],
            overlay,
            frame
        ), alpha=0.45)

    ax.set_title(f"Frame {idx}")
    ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()

In [ ]:
import os
import json
import zipfile
import pandas as pd

output_dir = "/content/FastReasonSeg_Results"
os.makedirs(output_dir, exist_ok=True)

# Collect metrics from existing results
valid_masks = int((tracking_df["status"] == "Valid mask").sum())
empty_masks = int((tracking_df["status"] == "Empty mask").sum())
missing_masks = int((tracking_df["status"] == "Missing mask").sum())

mean_iou = float(iou_df["consecutive_frame_iou"].mean())
median_iou = float(iou_df["consecutive_frame_iou"].median())
min_iou = float(iou_df["consecutive_frame_iou"].min())

report = f"""
FastReasonSeg Prototype — Video Evaluation Report
=================================================

Video
-----
File: 01_dog.mp4
Resolution: 1280 x 720
Frames: {len(tracking_df)}

Video Segmentation
------------------
Frames processed: {len(video_results)}
Frames with nonempty masks: {valid_masks}
Frames with empty masks: {empty_masks}
Frames with missing masks: {missing_masks}
Mask availability: {valid_masks / len(tracking_df) * 100:.2f}%

Temporal Stability
------------------
Valid consecutive-frame comparisons: {len(iou_df)}
Mean consecutive-frame IoU: {mean_iou:.4f}
Median consecutive-frame IoU: {median_iou:.4f}
Minimum consecutive-frame IoU: {min_iou:.4f}

Lowest-IoU Transitions
----------------------
{iou_df.nsmallest(5, 'consecutive_frame_iou').to_string(index=False)}

Limitations
-----------
1. Temporal IoU measures overlap between consecutive predicted masks.
2. Temporal IoU is not ground-truth segmentation accuracy.
3. No ground-truth masks were used in these calculations.
4. Empty masks occur in some frames and require further investigation.
5. These results demonstrate a SAM2-based video-segmentation prototype,
   not a verified full reproduction of the FastReasonSeg research paper.
"""

report_path = os.path.join(output_dir, "evaluation_report.txt")

with open(report_path, "w") as f:
    f.write(report.strip() + "\n")

# Update summary JSON
summary_path = os.path.join(output_dir, "evaluation_summary.json")

if os.path.exists(summary_path):
    with open(summary_path, "r") as f:
        summary = json.load(f)
else:
    summary = {}

summary.update({
    "total_frames": int(len(tracking_df)),
    "valid_masks": valid_masks,
    "empty_masks": empty_masks,
    "missing_masks": missing_masks,
    "mask_availability_percent": round(
        valid_masks / len(tracking_df) * 100, 2
    ),
    "temporal_stability": {
        "valid_comparisons": int(len(iou_df)),
        "mean_iou": round(mean_iou, 4),
        "median_iou": round(median_iou, 4),
        "minimum_iou": round(min_iou, 4)
    },
    "ground_truth_evaluation": False
})

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(report)
print("\nReport saved:", report_path)
print("Summary updated:", summary_path)

In [ ]:
import os
import zipfile

zip_path = "/content/FastReasonSeg_Prototype_Results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, filenames in os.walk(output_dir):
        for filename in filenames:
            path = os.path.join(root, filename)
            zipf.write(path, os.path.relpath(path, output_dir))

with zipfile.ZipFile(zip_path, "r") as zipf:
    print("ZIP integrity:", "PASS" if zipf.testzip() is None else "FAIL")
    print("\nContents:")
    for name in zipf.namelist():
        print("-", name)

print("\nZIP size (MB):", round(os.path.getsize(zip_path) / (1024**2), 2))

In [ ]:
print("Digital Twin available:", "digital_twin" in globals())
print("Spatial reasoning function available:", "spatial_reasoning" in globals())
print("Spatial test results available:", "spatial_test_results" in globals())
print("Video results available:", "video_results" in globals())

if "digital_twin" in globals():
    print("\nDigital Twin type:", type(digital_twin))
    if isinstance(digital_twin, dict):
        print("Digital Twin keys:", list(digital_twin.keys())[:20])

In [ ]:
import json
import os

notebook_path = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "FastReasonSeg_Project.ipynb"
)

if not os.path.exists(notebook_path):
    print("Original notebook not found:", notebook_path)
else:
    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    keywords = [
        "digital twin",
        "spatial reasoning",
        "spatial_reasoning",
        "class DigitalTwin",
        "depth anything",
        "relative position",
        "left_of",
        "right_of",
    ]

    for i, cell in enumerate(notebook.get("cells", [])):
        source = "".join(cell.get("source", []))

        if any(k.lower() in source.lower() for k in keywords):
            print(f"\n{'=' * 25} CELL {i} {'=' * 25}")
            print(source[:2500])

In [ ]:
import os

check_vars = [
    "image",
    "input_file",
    "owl_result",
    "predictor",
    "digital_twin_objects",
    "depth_map",
    "depth_model",
    "depth_processor",
    "device",
    "video_results",
]

for name in check_vars:
    exists = name in globals()
    print(f"{name}: {'AVAILABLE' if exists else 'MISSING'}")

print("\nSaved Digital Twin JSON exists:",
      os.path.exists("/content/digital_twin.json"))

In [ ]:
import os

from pathlib import Path

search_dirs = [
    "/content",
    "/content/drive/MyDrive/Colab Notebooks",
    "/content/drive/MyDrive",
]

extensions = {".jpg", ".jpeg", ".png", ".webp"}

for folder in search_dirs:
    if not os.path.exists(folder):
        print(f"\nNot available: {folder}")
        continue

    print(f"\n--- Images in {folder} ---")
    count = 0

    for path in Path(folder).rglob("*"):
        if path.is_file() and path.suffix.lower() in extensions:
            print(path)
            count += 1

            if count >= 30:
                print("Showing first 30 images only.")
                break

    if count == 0:
        print("No images found.")

In [ ]:
from PIL import Image
import os
import torch

image_path = "/content/first_frame.jpg"

print("Image exists:", os.path.exists(image_path))

if os.path.exists(image_path):
    img = Image.open(image_path)
    print("Image dimensions:", img.size)
    display(img)

print("\nGPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nSAM2 repository exists:", os.path.exists("/content/sam2"))
print("Video results available:", "video_results" in globals())
print("Video frames:", len(video_results) if "video_results" in globals() else 0)

In [ ]:
import os
import sys
import torch
import numpy as np
from PIL import Image

# Load image
image_path = "/content/first_frame.jpg"
image = Image.open(image_path).convert("RGB")
image_np = np.array(image)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Image:", image.size)
print("Device:", device)

# Restore SAM2 import
os.chdir("/tmp")
if "/content/sam2" not in sys.path:
    sys.path.insert(0, "/content/sam2")

import importlib

for name in list(sys.modules):
    if name == "sam2" or name.startswith("sam2."):
        del sys.modules[name]

importlib.invalidate_caches()

from sam2.sam2_image_predictor import SAM2ImagePredictor

print("SAM2 image predictor imported successfully.")

In [ ]:
import os
import sys
import torch
import numpy as np
from PIL import Image

# Load image
image_path = "/content/first_frame.jpg"
image = Image.open(image_path).convert("RGB")
image_np = np.array(image)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Image:", image.size)
print("Device:", device)

# Restore SAM2 import
os.chdir("/tmp")
if "/content/sam2" not in sys.path:
    sys.path.insert(0, "/content/sam2")

import importlib

for name in list(sys.modules):
    if name == "sam2" or name.startswith("sam2."):
        del sys.modules[name]

importlib.invalidate_caches()

from sam2.sam2_image_predictor import SAM2ImagePredictor

print("SAM2 image predictor imported successfully.")

In [ ]:
import os
import torch

checkpoint_candidates = [
    "/content/checkpoints/sam2.1_hiera_small.pt",
    "/content/sam2/checkpoints/sam2.1_hiera_small.pt",
    "/content/sam2/checkpoints/sam2.1_hiera_tiny.pt",
]

checkpoint_path = next(
    (p for p in checkpoint_candidates if os.path.exists(p)),
    None
)

if checkpoint_path is None:
    print("SAM2 checkpoint not found in the standard locations.")
    print("Searching /content for SAM2 checkpoints...")

    import glob
    matches = glob.glob("/content/**/*.pt", recursive=True)
    matches = [p for p in matches if "sam2" in p.lower()]

    for p in matches[:20]:
        print(p)
else:
    print("Checkpoint found:", checkpoint_path)

In [ ]:
import os
import urllib.request

checkpoint_dir = "/content/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(
    checkpoint_dir, "sam2.1_hiera_small.pt"
)

checkpoint_url = (
    "https://dl.fbaipublicfiles.com/segment_anything_2/"
    "072824/sam2.1_hiera_small.pt"
)

if os.path.exists(checkpoint_path):
    print("Checkpoint already exists:", checkpoint_path)
else:
    print("Downloading SAM2.1 Small checkpoint...")

    try:
        urllib.request.urlretrieve(
            checkpoint_url, checkpoint_path
        )

        if os.path.getsize(checkpoint_path) < 1_000_000:
            raise RuntimeError(
                "Downloaded file is unexpectedly small."
            )

        print("Download successful.")
        print(
            "Size:",
            round(os.path.getsize(checkpoint_path) / (1024**2), 2),
            "MB"
        )
        print("Path:", checkpoint_path)

    except Exception as e:
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
        print("Download failed:", repr(e))

In [ ]:
import os
import urllib.request

checkpoint_dir = "/content/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(
    checkpoint_dir, "sam2.1_hiera_small.pt"
)

checkpoint_url = (
    "https://huggingface.co/facebook/sam2.1-hiera-small/"
    "resolve/main/sam2.1_hiera_small.pt"
)

try:
    print("Downloading SAM2.1 Small from Hugging Face...")

    request = urllib.request.Request(
        checkpoint_url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urllib.request.urlopen(request, timeout=180) as response:
        with open(checkpoint_path, "wb") as f:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)

    size_mb = os.path.getsize(checkpoint_path) / (1024 ** 2)

    if size_mb < 100:
        raise RuntimeError(
            f"Checkpoint looks incomplete: only {size_mb:.1f} MB"
        )

    print("Download completed.")
    print(f"Checkpoint size: {size_mb:.1f} MB")
    print("Saved to:", checkpoint_path)

except Exception as e:
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    print("Download failed:", repr(e))

In [ ]:
import torch
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.build_sam import build_sam2

config_file = "configs/sam2.1/sam2.1_hiera_s.yaml"
checkpoint_path = "/content/checkpoints/sam2.1_hiera_small.pt"

print("Loading SAM2.1 Small...")

sam2_model = build_sam2(
    config_file,
    checkpoint_path,
    device="cuda"
)

predictor = SAM2ImagePredictor(sam2_model)

print("SAM2.1 Small loaded successfully!")

# Set image for segmentation
predictor.set_image(image_np)

print("Image loaded into SAM2.")
print("Image shape:", image_np.shape)
print("Video results preserved:", len(video_results), "frames")

In [ ]:
# Install/import OWLv2 dependencies
!pip -q install transformers

import torch
import numpy as np
from transformers import AutoProcessor, Owlv2ForObjectDetection

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/owlv2-base-patch16-ensemble"

print("Loading OWLv2...")

owl_processor = AutoProcessor.from_pretrained(model_id)
owl_model = Owlv2ForObjectDetection.from_pretrained(
    model_id
).to(device)

owl_model.eval()

print("OWLv2 loaded successfully.")

# Detect common objects
candidate_labels = [
    "dog",
    "person",
    "car",
    "bicycle",
    "truck",
    "chair",
    "cat",
    "tree",
    "ball",
]

inputs = owl_processor(
    text=[candidate_labels],
    images=image,
    return_tensors="pt"
).to(device)

with torch.inference_mode():
    outputs = owl_model(**inputs)

target_sizes = torch.tensor(
    [[image.height, image.width]],
    device=device
)

results = owl_processor.post_process_grounded_object_detection(
    outputs=outputs,
    target_sizes=target_sizes,
    threshold=0.15,
    text_labels=[candidate_labels]
)

owl_result = results[0]

print("\nDetection complete.")
print("Detected objects:", len(owl_result["boxes"]))

for box, score, label in zip(
    owl_result["boxes"],
    owl_result["scores"],
    owl_result["text_labels"]
):
    print(
        f"{label}: confidence={score.item():.3f}, "
        f"box={[round(x, 1) for x in box.tolist()]}"
    )

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Convert OWLv2 detections to CPU NumPy
boxes = owl_result["boxes"].detach().cpu().numpy()
scores = owl_result["scores"].detach().cpu().numpy()
labels = list(owl_result["text_labels"])

# Non-maximum suppression to remove duplicate boxes
def nms(boxes, scores, threshold=0.5):
    if len(boxes) == 0:
        return []

    order = np.argsort(scores)[::-1]
    keep = []

    while len(order) > 0:
        i = order[0]
        keep.append(i)

        if len(order) == 1:
            break

        rest = order[1:]
        xx1 = np.maximum(boxes[i, 0], boxes[rest, 0])
        yy1 = np.maximum(boxes[i, 1], boxes[rest, 1])
        xx2 = np.minimum(boxes[i, 2], boxes[rest, 2])
        yy2 = np.minimum(boxes[i, 3], boxes[rest, 3])

        intersection = (
            np.maximum(0, xx2 - xx1) *
            np.maximum(0, yy2 - yy1)
        )

        area_i = (
            (boxes[i, 2] - boxes[i, 0]) *
            (boxes[i, 3] - boxes[i, 1])
        )
        area_rest = (
            (boxes[rest, 2] - boxes[rest, 0]) *
            (boxes[rest, 3] - boxes[rest, 1])
        )

        iou = intersection / np.maximum(
            area_i + area_rest - intersection, 1e-8
        )

        order = rest[iou <= threshold]

    return keep

keep = nms(boxes, scores, threshold=0.5)

digital_twin_objects = []

for idx in keep:
    box = boxes[idx].astype(np.float32)

    with torch.inference_mode():
        masks, mask_scores, _ = predictor.predict(
            box=box,
            multimask_output=True
        )

    best_idx = int(np.argmax(mask_scores))
    mask = np.asarray(masks[best_idx]).astype(bool)

    ys, xs = np.where(mask)

    if len(xs) == 0:
        continue

    digital_twin_objects.append({
        "id": len(digital_twin_objects) + 1,
        "label": str(labels[idx]),
        "confidence": float(scores[idx]),
        "bbox": box.tolist(),
        "mask": mask,
        "area": int(mask.sum()),
        "center": [float(xs.mean()), float(ys.mean())],
        "mask_area_ratio": float(mask.mean()),
        "bbox_width": float(box[2] - box[0]),
        "bbox_height": float(box[3] - box[1])
    })

print("Segmented objects:", len(digital_twin_objects))

for obj in digital_twin_objects:
    print(
        f'{obj["id"]}. {obj["label"]}: '
        f'confidence={obj["confidence"]:.3f}, '
        f'mask area={obj["area"]} pixels'
    )

# Visualize masks and bounding boxes
plt.figure(figsize=(14, 8))
plt.imshow(image_np)

for obj in digital_twin_objects:
    x1, y1, x2, y2 = obj["bbox"]
    plt.contour(obj["mask"], levels=[0.5], linewidths=1.5)
    plt.gca().add_patch(Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        fill=False, linewidth=1.5
    ))
    plt.text(
        x1, max(0, y1 - 5),
        f'{obj["label"]} ({obj["confidence"]:.2f})',
        color="yellow", fontsize=10,
        bbox=dict(facecolor="black", alpha=0.6)
    )

plt.axis("off")
plt.title("OWLv2 Detection + SAM2 Segmentation")
plt.show()

In [ ]:
print("Video results available:", "video_results" in globals())

if "video_results" in globals():
    print("Frames processed:", len(video_results))
    print("First frame:", min(video_results) if video_results else "No frames")
    print("Last frame:", max(video_results) if video_results else "No frames")
else:
    print("Video propagation has not run in this runtime.")

In [ ]:
# Check current project state without changing anything

import torch
import os

variables_to_check = [
    "video_results",
    "image",
    "image_np",
    "owl_result",
    "predictor",
    "digital_twin_objects",
    "depth_map",
    "digital_twin"
]

for name in variables_to_check:
    exists = name in globals()
    print(f"{name}: {'AVAILABLE' if exists else 'MISSING'}")

if "video_results" in globals():
    print("\nVideo results available:", video_results is not None)
    print("Frames processed:", len(video_results))
    print("First frame:", min(video_results.keys()))
    print("Last frame:", max(video_results.keys()))

print("\nGPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import json
import os
import numpy as np

# Check required variables
assert "image" in globals(), "Image is missing"
assert "digital_twin_objects" in globals(), "Segmented objects are missing"

# Create a JSON-serializable Digital Twin
digital_twin = {
    "image": "/content/first_frame.jpg",
    "width": int(image.width),
    "height": int(image.height),
    "num_objects": len(digital_twin_objects),
    "objects": []
}

# Extract object information (exclude NumPy masks)
for obj in digital_twin_objects:
    digital_twin["objects"].append({
        "id": int(obj["id"]),
        "label": str(obj["label"]),
        "confidence": float(obj["confidence"]),
        "bbox": [float(x) for x in obj["bbox"]],
        "area": int(obj["area"]),
        "center": [float(x) for x in obj["center"]],
        "mask_area_ratio": float(obj["mask_area_ratio"]),
        "bbox_width": float(obj["bbox_width"]),
        "bbox_height": float(obj["bbox_height"])
    })

# Save to disk
output_path = "/content/digital_twin.json"

with open(output_path, "w") as f:
    json.dump(digital_twin, f, indent=2)

print("Digital Twin created successfully!")
print("Image dimensions:", digital_twin["width"], "x", digital_twin["height"])
print("Objects:", digital_twin["num_objects"])
print("Saved to:", output_path)

for obj in digital_twin["objects"]:
    print(
        f'{obj["id"]}. {obj["label"]} | '
        f'Confidence: {obj["confidence"]:.3f} | '
        f'Area: {obj["area"]} pixels'
    )

In [ ]:
import json
import math

# Load the saved Digital Twin
with open("/content/digital_twin.json", "r") as f:
    digital_twin = json.load(f)

objects = digital_twin["objects"]

# Find an object by label
def find_object(label):
    matches = [
        obj for obj in objects
        if label.lower() in obj["label"].lower()
    ]
    return matches[0] if matches else None


# Find the object with the largest segmentation area
def largest_object():
    return max(objects, key=lambda obj: obj["area"]) if objects else None


# Determine relative position using object centers
def spatial_relation(label1, label2):
    obj1 = find_object(label1)
    obj2 = find_object(label2)

    if obj1 is None or obj2 is None:
        return "Object not found."

    x1, y1 = obj1["center"]
    x2, y2 = obj2["center"]

    relations = []

    if x1 < x2:
        relations.append(f"{obj1['label']} is left of {obj2['label']}")
    elif x1 > x2:
        relations.append(f"{obj1['label']} is right of {obj2['label']}")

    if y1 < y2:
        relations.append(f"{obj1['label']} is above {obj2['label']}")
    elif y1 > y2:
        relations.append(f"{obj1['label']} is below {obj2['label']}")

    distance = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    return {
        "relation": "; ".join(relations) if relations else "Centers coincide",
        "center_distance_pixels": round(distance, 2)
    }


# Test 1: Largest object
largest = largest_object()
print("1. Largest object:")
print(largest["label"] if largest else "No objects found")

# Test 2: Dog relative to person
print("\n2. Dog relative to person:")
print(spatial_relation("dog", "person"))

# Test 3: Ball relative to dog
print("\n3. Ball relative to dog:")
print(spatial_relation("ball", "dog"))

# Test 4: List all detected objects
print("\n4. Objects in Digital Twin:")
for obj in objects:
    print(f'{obj["label"]}: center={obj["center"]}, area={obj["area"]}')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from PIL import Image

# 1. Load the depth estimation model
model_id = "depth-anything/Depth-Anything-V2-Small-hf"

depth_processor = AutoImageProcessor.from_pretrained(model_id)
depth_model = AutoModelForDepthEstimation.from_pretrained(model_id)
depth_model = depth_model.to(device).eval()

print("Depth model loaded successfully.")

# 2. Estimate relative depth
inputs = depth_processor(images=image, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.inference_mode():
    outputs = depth_model(**inputs)

# Resize the predicted depth map to the original image dimensions
depth_map = torch.nn.functional.interpolate(
    outputs.predicted_depth.unsqueeze(1),
    size=(image.height, image.width),
    mode="bicubic",
    align_corners=False
).squeeze().cpu().numpy()

print("Depth map shape:", depth_map.shape)
print("Depth range:", float(depth_map.min()), "to", float(depth_map.max()))

# 3. Save depth map
os.makedirs("/content/FastReasonSeg_Results", exist_ok=True)

np.save("/content/FastReasonSeg_Results/depth_map.npy", depth_map)

# 4. Visualize estimated relative depth
plt.figure(figsize=(12, 6))
plt.imshow(depth_map, cmap="plasma")
plt.colorbar(label="Relative depth prediction")
plt.title("Depth Anything V2 — Relative Depth Estimate")
plt.axis("off")
plt.show()

# 5. Calculate depth statistics inside each segmentation mask
print("\nObject depth statistics:")

for obj in digital_twin_objects:
    mask = np.asarray(obj["mask"]).astype(bool)

    if mask.shape != depth_map.shape:
        print(f'{obj["label"]}: mask shape mismatch; skipped')
        continue

    values = depth_map[mask]
    values = values[np.isfinite(values)]

    if len(values) == 0:
        print(f'{obj["label"]}: no valid depth values')
        continue

    obj["mean_depth"] = float(np.mean(values))
    obj["median_depth"] = float(np.median(values))

    print(
        f'{obj["label"]}: '
        f'mean={obj["mean_depth"]:.4f}, '
        f'median={obj["median_depth"]:.4f}'
    )

print("\nSaved depth map to /content/FastReasonSeg_Results/depth_map.npy")

In [ ]:
import json
import os
import numpy as np

output_dir = "/content/FastReasonSeg_Results"
os.makedirs(output_dir, exist_ok=True)

# Add depth statistics to the Digital Twin
for obj in digital_twin["objects"]:
    matching = next(
        (x for x in digital_twin_objects if x["id"] == obj["id"]),
        None
    )

    if matching is not None and "mean_depth" in matching:
        obj["mean_relative_depth"] = matching["mean_depth"]
        obj["median_relative_depth"] = matching["median_depth"]

# Record model and measurement limitations
digital_twin["depth_model"] = "Depth Anything V2 Small"
digital_twin["depth_units"] = "relative model output; not metres"

# Save updated Digital Twin
json_path = os.path.join(output_dir, "digital_twin.json")

with open(json_path, "w") as f:
    json.dump(digital_twin, f, indent=2)

# Save masks separately because NumPy arrays cannot be stored directly in JSON
mask_path = os.path.join(output_dir, "segmentation_masks.npz")

np.savez_compressed(
    mask_path,
    **{
        f"object_{obj['id']}_{obj['label']}": np.asarray(obj["mask"], dtype=np.uint8)
        for obj in digital_twin_objects
    }
)

# Verify saved JSON
with open(json_path, "r") as f:
    verified_twin = json.load(f)

print("Digital Twin updated and verified.")
print("Objects:", verified_twin["num_objects"])
print("Depth statistics:", sum(
    "mean_relative_depth" in obj for obj in verified_twin["objects"]
), "/", len(verified_twin["objects"]))
print("JSON:", json_path)
print("Masks:", mask_path)
print("Depth map:", os.path.join(output_dir, "depth_map.npy"))

for obj in verified_twin["objects"]:
    print(
        f"{obj['label']}: "
        f"mean relative depth = "
        f"{obj.get('mean_relative_depth', 'unavailable')}"
    )

In [ ]:
import os
import json
import zipfile
import shutil

BASE = "/content/FastReasonSeg_Results"
os.makedirs(BASE, exist_ok=True)

# Copy existing video-result artifacts when available
old_zip = "/content/FastReasonSeg_Prototype_Results.zip"
video_dir = os.path.join(BASE, "video_results")
os.makedirs(video_dir, exist_ok=True)

if os.path.exists(old_zip):
    with zipfile.ZipFile(old_zip, "r") as z:
        z.extractall(video_dir)
    print("Existing video results included.")
else:
    print("Previous video ZIP not found; retaining current runtime results.")

# Record implementation status honestly
report = {
    "project": "FastReasonSeg-Inspired Image and Video Segmentation",
    "image_pipeline": {
        "object_detection": "OWLv2",
        "segmentation": "SAM2.1 Small",
        "depth_estimation": "Depth Anything V2 Small",
        "detected_objects": 5,
        "digital_twin": "Saved",
        "spatial_reasoning": "Basic center-based relations implemented"
    },
    "video_pipeline": {
        "model": "SAM2.1 Small",
        "frames_processed": 289,
        "first_frame": 0,
        "last_frame": 288,
        "ground_truth_evaluation": False
    },
    "limitations": [
        "Results use a prototype pipeline and have not been validated against ground-truth masks.",
        "Depth predictions are relative model outputs, not measurements in metres.",
        "Basic spatial reasoning is implemented; comprehensive reasoning evaluation remains incomplete.",
        "Qwen3-8B teacher integration, supervised fine-tuning, GRPO training, and student distillation are not implemented.",
        "The implementation is not a complete reproduction of the FastReasonSeg research paper."
    ]
}

with open(os.path.join(BASE, "project_evaluation.json"), "w") as f:
    json.dump(report, f, indent=2)

# Create a readable README
readme = """# FastReasonSeg-Inspired Segmentation Prototype

## Implemented
- OWLv2 object detection
- SAM2.1 Small image segmentation
- Depth Anything V2 relative-depth estimation
- Digital Twin object metadata
- Basic center-based spatial relations
- SAM2 video segmentation over 289 frames

## Main artifacts
- digital_twin.json: object metadata and depth statistics
- segmentation_masks.npz: segmentation masks
- depth_map.npy: relative-depth map
- project_evaluation.json: implementation status and limitations
- video_results/: previously generated video evaluation artifacts, if available

## Limitations
This is a prototype inspired by FastReasonSeg, not a complete paper reproduction.
Ground-truth segmentation accuracy has not been established.
Depth values are not physical distances.
Qwen3-8B teacher integration, SFT, GRPO, and student distillation are not implemented.
"""

with open(os.path.join(BASE, "README.txt"), "w") as f:
    f.write(readme)

# Include original video output files directly when they exist
for filename in [
    "segmented_dog.mp4",
    "tracking_quality_plot.png",
    "tracking_metrics.csv",
    "temporal_stability.csv",
    "evaluation_summary.json",
    "evaluation_report.txt",
    "empty_frames.json"
]:
    source = os.path.join("/content", filename)
    if os.path.exists(source):
        shutil.copy2(source, BASE)

# Create final ZIP
final_zip = "/content/FastReasonSeg_Final_Project.zip"

with zipfile.ZipFile(final_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(BASE):
        for filename in files:
            full_path = os.path.join(root, filename)
            relative_path = os.path.relpath(full_path, BASE)
            z.write(full_path, relative_path)

# Verify ZIP integrity
with zipfile.ZipFile(final_zip, "r") as z:
    bad_file = z.testzip()
    print("\nZIP integrity:", "PASS" if bad_file is None else f"FAIL: {bad_file}")
    print("Files included:", len(z.namelist()))

print("Final ZIP:", final_zip)
print("Size: {:.2f} MB".format(os.path.getsize(final_zip) / (1024 * 1024)))

In [ ]:
from google.colab import files
files.download("/content/FastReasonSeg_Final_Project.zip")

In [ ]:
# FASTREASONSEG: QUANTITATIVE VIDEO EVALUATION
# Reuses existing evaluation files; does not rerun segmentation.

import os
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASE = "/content/FastReasonSeg_Results"
OUT = os.path.join(BASE, "quantitative_evaluation")
os.makedirs(OUT, exist_ok=True)

# 1. Find previously saved metrics
search_dirs = [
    BASE,
    "/content",
    "/content/video_results"
]

csv_files = []
for folder in search_dirs:
    if os.path.isdir(folder):
        csv_files.extend(
            glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
        )

csv_files = sorted(set(csv_files))

print("CSV files found:")
for f in csv_files:
    print(" -", f)

# Load tracking/temporal metric tables
tables = {}
for path in csv_files:
    try:
        df = pd.read_csv(path)
        if not df.empty:
            tables[os.path.basename(path)] = (path, df)
    except Exception as e:
        print(f"Could not read {path}: {e}")

print("\nAvailable tables:", list(tables.keys()))

# 2. Display each metrics table's columns and sample rows
for name, (path, df) in tables.items():
    print(f"\n--- {name} ---")
    print("Columns:", list(df.columns))
    display(df.head())

# 3. Load saved summary/report, if available
for filename in ["evaluation_summary.json", "project_evaluation.json"]:
    matches = glob.glob(
        os.path.join("/content", "**", filename), recursive=True
    )
    for path in sorted(set(matches)):
        try:
            with open(path, "r") as f:
                data = json.load(f)
            print(f"\n--- {path} ---")
            print(json.dumps(data, indent=2)[:6000])
        except Exception as e:
            print("Could not read summary:", e)

# 4. Find likely columns automatically
def find_column(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    for c in df.columns:
        cl = str(c).strip().lower()
        if any(candidate in cl for candidate in candidates):
            return c
    return None

temporal_item = next(
    (
        (name, path, df)
        for name, (path, df) in tables.items()
        if "temporal" in name.lower()
        or any("iou" in str(c).lower() for c in df.columns)
    ),
    None
)

tracking_item = next(
    (
        (name, path, df)
        for name, (path, df) in tables.items()
        if "tracking" in name.lower()
    ),
    None
)

# 5. Plot temporal IoU if the table contains it
if temporal_item:
    name, path, df = temporal_item
    iou_col = find_column(
        df, ["temporal_iou", "iou", "consecutive_iou", "mask_iou"]
    )
    frame_col = find_column(
        df, ["frame", "frame_idx", "frame_index", "transition", "index"]
    )

    if iou_col:
        plot_df = df.copy()
        plot_df[iou_col] = pd.to_numeric(
            plot_df[iou_col], errors="coerce"
        )
        plot_df = plot_df.dropna(subset=[iou_col])

        if len(plot_df):
            x = (
                pd.to_numeric(plot_df[frame_col], errors="coerce")
                if frame_col else np.arange(len(plot_df))
            )

            plt.figure(figsize=(12, 4))
            plt.plot(x, plot_df[iou_col], linewidth=1)
            plt.axhline(
                plot_df[iou_col].mean(),
                linestyle="--",
                label=f"Mean = {plot_df[iou_col].mean():.3f}"
            )
            plt.xlabel("Frame / transition")
            plt.ylabel("Temporal IoU")
            plt.title("Consecutive-frame Mask Consistency")
            plt.ylim(0, 1.05)
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.tight_layout()

            graph_path = os.path.join(
                OUT, "temporal_iou_over_frames.png"
            )
            plt.savefig(graph_path, dpi=160)
            plt.show()
            print("Saved:", graph_path)

            print("\nLowest temporal-IoU transitions:")
            display(
                plot_df.sort_values(iou_col).head(10)
            )
    else:
        print("No recognizable IoU column in:", name)

# 6. Plot mask area / tracking values if available
if tracking_item:
    name, path, df = tracking_item
    print(f"\nTracking table selected: {name}")

    frame_col = find_column(
        df, ["frame", "frame_idx", "frame_index", "index"]
    )
    area_col = find_column(
        df, ["mask_area", "area", "pixel_count", "mask_pixels"]
    )

    if area_col:
        plot_df = df.copy()
        plot_df[area_col] = pd.to_numeric(
            plot_df[area_col], errors="coerce"
        )
        plot_df = plot_df.dropna(subset=[area_col])
        x = (
            pd.to_numeric(plot_df[frame_col], errors="coerce")
            if frame_col else np.arange(len(plot_df))
        )

        plt.figure(figsize=(12, 4))
        plt.plot(x, plot_df[area_col], linewidth=1)
        plt.xlabel("Frame")
        plt.ylabel("Mask area (pixels)")
        plt.title("Segmentation Mask Area Across Video")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        graph_path = os.path.join(
            OUT, "mask_area_over_frames.png"
        )
        plt.savefig(graph_path, dpi=160)
        plt.show()
        print("Saved:", graph_path)
    else:
        print("No recognizable mask-area column in tracking table.")

# 7. Create a concise evaluation note using existing known measurements
report = {
    "video_frames": 289,
    "nonempty_mask_frames": 266,
    "empty_mask_frames": 23,
    "missing_mask_frames": 0,
    "mask_availability_percent": round(266 / 289 * 100, 2),
    "temporal_iou_valid_comparisons": 264,
    "mean_temporal_iou": 0.753,
    "median_temporal_iou": 0.7675,
    "minimum_temporal_iou": 0.0481,
    "lowest_iou_transition_frame": 77,
    "interpretation": (
        "Temporal IoU measures consecutive-mask consistency, not "
        "ground-truth segmentation accuracy. Object motion and scene "
        "changes can lower it. Ground-truth masks are required for "
        "standard accuracy metrics such as mIoU."
    )
}

report_path = os.path.join(OUT, "quantitative_evaluation_summary.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("\n--- QUANTITATIVE EVALUATION SUMMARY ---")
print(json.dumps(report, indent=2))
print("\nEvaluation output folder:", OUT)
print("Summary saved:", report_path)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = "/content/FastReasonSeg_Results/tracking_metrics.csv"
video_path = "/content/01_dog.mp4"

df = pd.read_csv(metrics_path)

# Find frames where the segmentation mask is empty
empty_frames = df.loc[
    pd.to_numeric(df["mask_pixels"], errors="coerce").fillna(0) == 0,
    "frame"
].astype(int).tolist()

print(f"Empty-mask frames: {len(empty_frames)}")
print("Frame numbers:", empty_frames)

# Inspect frames around the largest temporal-IoU drop
inspect_frames = sorted(set(
    list(range(72, 80)) + list(range(100, 104))
))

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")

frames = []

for frame_id in inspect_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
    ok, frame = cap.read()

    if ok:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append((frame_id, frame))

cap.release()

if not frames:
    raise RuntimeError("No frames could be read.")

cols = 4
rows = int(np.ceil(len(frames) / cols))

fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = np.array(axes).reshape(-1)

for ax, (frame_id, frame) in zip(axes, frames):
    ax.imshow(frame)
    ax.set_title(f"Video frame {frame_id}")
    ax.axis("off")

for ax in axes[len(frames):]:
    ax.axis("off")

plt.suptitle("Visual inspection of low temporal-IoU regions")
plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Paths
metrics_path = "/content/FastReasonSeg_Results/tracking_metrics.csv"
video_path = "/content/01_dog.mp4"

# Check files
assert os.path.exists(metrics_path), f"Missing: {metrics_path}"
assert os.path.exists(video_path), f"Missing: {video_path}"

# Load tracking metrics
df = pd.read_csv(metrics_path)

# Find frames with empty masks
df["mask_pixels"] = pd.to_numeric(
    df["mask_pixels"], errors="coerce"
).fillna(0)

empty_frames = sorted(
    df.loc[df["mask_pixels"] == 0, "frame"].astype(int).tolist()
)

print("Total frames in metrics:", len(df))
print("Empty-mask frames:", len(empty_frames))
print("Empty frame indices:", empty_frames)

# Inspect frames around the suspicious transition at 73–78
# and the later frames around 100–103.
inspect_frames = sorted(
    set(list(range(70, 81)) + list(range(99, 105)))
)

cap = cv2.VideoCapture(video_path)
frames = []

for frame_idx in inspect_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    success, frame = cap.read()

    if success:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append((frame_idx, frame))

cap.release()

if frames:
    cols = 4
    rows = (len(frames) + cols - 1) // cols

    fig, axes = plt.subplots(
        rows, cols, figsize=(16, 4 * rows)
    )
    axes = axes.flatten()

    for ax, (idx, frame) in zip(axes, frames):
        ax.imshow(frame)
        ax.set_title(f"Frame {idx}")
        ax.axis("off")

    for ax in axes[len(frames):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No video frames could be read.")

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = "/content/01_dog.mp4"

# Frames before, during, and after the failure
inspect_frames = [70, 74, 77, 78, 82, 86, 90, 94, 98, 100, 101, 105, 110]

cap = cv2.VideoCapture(video_path)

frames = []

for idx in inspect_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    success, frame = cap.read()

    if success:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append((idx, frame))

cap.release()

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for ax, (idx, frame) in zip(axes, frames):
    ax.imshow(frame)
    ax.set_title(f"Frame {idx}")
    ax.axis("off")

for ax in axes[len(frames):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

video_path = "/content/01_dog.mp4"
mask_path = "/content/FastReasonSeg_Results/segmentation_masks.npz"

print("Mask file exists:", os.path.exists(mask_path))

data = np.load(mask_path)
print("Saved arrays:", data.files)

for key in data.files:
    arr = data[key]
    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

In [ ]:
import os
import numpy as np

# 1. Inspect the saved dog mask
mask_path = "/content/FastReasonSeg_Results/segmentation_masks.npz"
data = np.load(mask_path)

dog_mask = data["object_1_dog"]

print("DOG MASK")
print("Shape:", dog_mask.shape)
print("Unique values:", np.unique(dog_mask))
print("Foreground pixels:", np.count_nonzero(dog_mask))
print("Mask area (%):", round(np.count_nonzero(dog_mask) / dog_mask.size * 100, 3))

# 2. Inspect saved video results
video_dir = "/content/FastReasonSeg_Results/video_results"

print("\nVIDEO RESULTS DIRECTORY")
print("Directory exists:", os.path.isdir(video_dir))

if os.path.isdir(video_dir):
    for name in sorted(os.listdir(video_dir))[:30]:
        path = os.path.join(video_dir, name)

        if os.path.isfile(path):
            print(name, "-", os.path.getsize(path), "bytes")
        else:
            print(name, "- directory")

In [ ]:
print("Hello Charan")

In [ ]:
import pyarrow
import torch

print("PyArrow:", pyarrow.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import datasets
print("Datasets:", datasets.__version__)
print("Datasets imported successfully!")

In [ ]:
import torch
import pyarrow
import datasets
import transformers
import trl
import peft

print("PyTorch:", torch.__version__)
print("PyArrow:", pyarrow.__version__)
print("Datasets:", datasets.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import json

RESULTS_DIR = "/content/FastReasonSeg_Results"

files_to_check = [
    "digital_twin_enriched.json",
    "verified_spatial_answer.json",
    "spatial_sft_dataset.jsonl",
    "segmentation_masks.npz"
]

for filename in files_to_check:
    path = os.path.join(RESULTS_DIR, filename)
    print(f"{filename}: {'EXISTS' if os.path.exists(path) else 'MISSING'}")

In [ ]:
import json
import os

dataset_path = "/content/FastReasonSeg_Results/spatial_sft_dataset.jsonl"

examples = []

with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            examples.append(json.loads(line))

print("Number of examples:", len(examples))

if examples:
    print("\nFirst example:")
    print(json.dumps(examples[0], indent=2, ensure_ascii=False))

In [ ]:
import torch
import transformers
import trl
import peft
import datasets

print("GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

print("SFTTrainer available:", hasattr(trl, "SFTTrainer"))
print("LoRA available:", hasattr(peft, "LoraConfig"))
print("Dataset available:", hasattr(datasets, "Dataset"))

In [ ]:
import inspect
import json
import os

from datasets import Dataset
from trl import SFTTrainer
from peft import LoraConfig

DATASET_PATH = "/content/FastReasonSeg_Results/spatial_sft_dataset.jsonl"

# Load the existing dataset
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    examples = [json.loads(line) for line in f if line.strip()]

dataset = Dataset.from_list(examples)

print("Dataset rows:", len(dataset))
print("Dataset columns:", dataset.column_names)
print("SFTTrainer parameters:")
print(inspect.signature(SFTTrainer.__init__))

print("\nLoRA configuration parameters:")
print(inspect.signature(LoraConfig))

In [ ]:
import os

paths_to_check = [
    "/content/FastReasonSeg_Results",
    "/root/.cache/huggingface/hub"
]

for path in paths_to_check:
    print(f"\n{path}")
    if os.path.exists(path):
        print(os.listdir(path)[:15])
    else:
        print("Directory not found")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    local_files_only=True
)

model = model.to("cuda")
model.config.use_cache = False

print("Model loaded successfully")
print("Device:", next(model.parameters()).device)
print("Tokenizer vocabulary:", len(tokenizer))
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

In [ ]:
for name, module in model.named_modules():
    if name.endswith(("q_proj", "k_proj", "v_proj", "o_proj")):
        print(name)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import importlib.metadata as metadata

for package in ["torch", "torchao", "peft", "transformers", "trl"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

In [ ]:
%pip uninstall -y torchao

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    local_files_only=True
).to("cuda")

model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import json
from datasets import Dataset

DATASET_PATH = "/content/FastReasonSeg_Results/spatial_sft_dataset.jsonl"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    examples = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(examples) > 0, "Dataset is empty!"

dataset = Dataset.from_list(examples)

print("Training examples:", len(dataset))
print("Example fields:", dataset.column_names)
print("First example:")
print(dataset[0]["messages"])

In [ ]:
import inspect
from trl import SFTConfig, SFTTrainer

print("SFTConfig supports max_length:",
      "max_length" in inspect.signature(SFTConfig).parameters)

print("SFTConfig supports max_seq_length:",
      "max_seq_length" in inspect.signature(SFTConfig).parameters)

In [ ]:
import os
import torch
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "/content/FastReasonSeg_Results/qwen3_spatial_lora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True,
    max_length=512,
    report_to="none",
    gradient_checkpointing=True,
    seed=42,
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Trainer initialized successfully.")
print("Training examples:", len(dataset))

In [ ]:
train_result = trainer.train()

print("\nTraining completed.")
print("Training metrics:")
print(train_result.metrics)

In [ ]:
ADAPTER_DIR = "/content/FastReasonSeg_Results/qwen3_spatial_lora/final_adapter"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("LoRA adapter saved to:", ADAPTER_DIR)
print("Adapter files:", os.listdir(ADAPTER_DIR))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-0.6B"
ADAPTER_DIR = (
    "/content/FastReasonSeg_Results/"
    "qwen3_spatial_lora/final_adapter"
)

tokenizer_ft = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    local_files_only=True
).to("cuda")

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

fine_tuned_model.eval()

print("Fine-tuned model loaded successfully.")

In [ ]:
import torch

question = "Where is the dog relative to the person?"

messages = [
    {
        "role": "user",
        "content": question
    }
]

prompt = tokenizer_ft.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer_ft(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output = fine_tuned_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer_ft.eos_token_id
    )

answer = tokenizer_ft.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Question:", question)
print("Fine-tuned answer:", answer)

In [ ]:
import torch

question = "Where is the dog relative to the person?"

messages = [
    {
        "role": "user",
        "content": question
    }
]

prompt = tokenizer_ft.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer_ft(
    prompt,
    return_tensors="pt"
).to("cuda")

def generate_answer(model, inputs, tokenizer):
    model.eval()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

print("BASE MODEL:")
print(generate_answer(base_model, inputs, tokenizer_ft))

print("\nFINE-TUNED MODEL:")
print(generate_answer(fine_tuned_model, inputs, tokenizer_ft))

In [ ]:
import json

path = "/content/FastReasonSeg_Results/spatial_sft_dataset.jsonl"

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        example = json.loads(line)
        print(f"\nEXAMPLE {i}")
        for message in example["messages"]:
            print(f'{message["role"]}: {message["content"]}')

In [ ]:
import torch

def ask_model(model, question):
    messages = [{"role": "user", "content": question}]

    prompt = tokenizer_ft.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer_ft(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer_ft.eos_token_id
        )

    return tokenizer_ft.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

question = "Where is the dog relative to the person?"

print("BASE MODEL:")
print(ask_model(base_model, question))

print("\nFINE-TUNED MODEL:")
print(ask_model(fine_tuned_model, question))

In [ ]:
print("Training loss:", train_result.training_loss)
print("Training metrics:", train_result.metrics)

In [ ]:
scene_context = """
Scene spatial data:
Dog center: (591.03, 442.73)
Person center: (338.83, 372.86)
Image coordinates: x increases to the right; y increases downward.
"""

question = "Where is the dog relative to the person?"

messages = [
    {
        "role": "user",
        "content": scene_context + "\n" + question
    }
]

In [ ]:
scene_context = """
Scene spatial data:
Dog center: (591.03, 442.73)
Person center: (338.83, 372.86)
Image coordinates: x increases to the right; y increases downward.
"""

question = "Where is the dog relative to the person?"

messages = [
    {
        "role": "user",
        "content": scene_context + "\n" + question
    }
]

print("Scene context:")
print(scene_context)

print("Question:")
print(question)

print("\nComplete model input:")
print(messages)

In [ ]:
answer = ask_model(
    fine_tuned_model,
    scene_context + "\n" + question
)

print("Fine-tuned model answer:")
print(answer)

In [ ]:
import torch

scene_context = """
Scene spatial data:
Dog center: (591.03, 442.73)
Person center: (338.83, 372.86)
Image coordinates: x increases to the right; y increases downward.
"""

question = "Where is the dog relative to the person?"
prompt_text = scene_context + "\n" + question

def compare_models(model, tokenizer, prompt_text):
    messages = [{"role": "user", "content": prompt_text}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(next(model.parameters()).device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

print("BASE MODEL ANSWER:")
print(compare_models(base_model, tokenizer_ft, prompt_text))

print("\nFINE-TUNED MODEL ANSWER:")
print(compare_models(fine_tuned_model, tokenizer_ft, prompt_text))

In [ ]:
test_cases = [
    {
        "context": "Dog center: (200, 100). Person center: (500, 300).",
        "question": "Where is the dog relative to the person?",
        "expected": "left and above"
    },
    {
        "context": "Dog center: (600, 500). Person center: (300, 200).",
        "question": "Where is the dog relative to the person?",
        "expected": "right and below"
    },
    {
        "context": "Dog center: (400, 150). Person center: (400, 350).",
        "question": "Where is the dog relative to the person?",
        "expected": "above"
    },
    {
        "context": "Dog center: (150, 400). Person center: (450, 400).",
        "question": "Where is the dog relative to the person?",
        "expected": "left"
    }
]

for i, case in enumerate(test_cases, 1):
    prompt = f"""
Image coordinate rules:
X increases to the right.
Y increases downward.

{case['context']}

Question: {case['question']}
Answer with the spatial relationship only.
"""
    print(f"\nTEST {i} | Expected: {case['expected']}")
    print("BASE:", compare_models(base_model, tokenizer_ft, prompt))
    print("FINE-TUNED:", compare_models(fine_tuned_model, tokenizer_ft, prompt))

In [ ]:
diagnostic_tests = [
    ("Dog (100, 100); Person (300, 300)", "left and above"),
    ("Dog (500, 100); Person (300, 300)", "right and above"),
    ("Dog (100, 500); Person (300, 300)", "left and below"),
    ("Dog (500, 500); Person (300, 300)", "right and below"),
]

for i, (coordinates, expected) in enumerate(diagnostic_tests, 1):
    prompt = f"""
Image coordinate rules:
X increases to the right.
Y increases downward.

{coordinates}

Identify the dog's position relative to the person.
Include both horizontal and vertical relationships.
"""
    print(f"\nTEST {i} | Expected: {expected}")
    print("BASE:", compare_models(base_model, tokenizer_ft, prompt))
    print("FINE-TUNED:", compare_models(fine_tuned_model, tokenizer_ft, prompt))



In [ ]:
def spatial_relationship(dog, person):
    dx = dog[0] - person[0]
    dy = dog[1] - person[1]

    if dx < 0:
        horizontal = "left"
    elif dx > 0:
        horizontal = "right"
    else:
        horizontal = "horizontally aligned"

    if dy < 0:
        vertical = "above"
    elif dy > 0:
        vertical = "below"
    else:
        vertical = "vertically aligned"

    return {
        "dx": dx,
        "dy": dy,
        "horizontal": horizontal,
        "vertical": vertical
    }


# Test
result = spatial_relationship((100, 100), (300, 300))
print(result)

In [ ]:
test_cases = [
    ((100, 100), (300, 300), "left", "above"),
    ((500, 100), (300, 300), "right", "above"),
    ((100, 500), (300, 300), "left", "below"),
    ((500, 500), (300, 300), "right", "below"),
]

correct = 0

for i, (dog, person, expected_x, expected_y) in enumerate(test_cases, 1):
    result = spatial_relationship(dog, person)

    passed = (
        result["horizontal"] == expected_x
        and result["vertical"] == expected_y
    )

    correct += passed

    print(
        f"Test {i}: {result['horizontal']} and {result['vertical']} | "
        f"{'PASS' if passed else 'FAIL'}"
    )

print(f"\nAccuracy: {correct}/{len(test_cases)} = "
      f"{correct / len(test_cases) * 100:.1f}%")

In [ ]:
import json

path = "/content/FastReasonSeg_Results/digital_twin_enriched.json"

with open(path, "r") as f:
    scene = json.load(f)

print(json.dumps(scene, indent=2)[:5000])


In [ ]:
import json
import itertools
import os

INPUT_PATH = "/content/FastReasonSeg_Results/digital_twin_enriched.json"
OUTPUT_PATH = "/content/FastReasonSeg_Results/computed_spatial_relationships.json"

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    scene = json.load(f)

def spatial_relationship(target, reference):
    tx, ty = target["center"]
    rx, ry = reference["center"]

    dx = tx - rx
    dy = ty - ry

    if dx < 0:
        horizontal = "left"
    elif dx > 0:
        horizontal = "right"
    else:
        horizontal = "horizontally aligned"

    if dy < 0:
        vertical = "above"
    elif dy > 0:
        vertical = "below"
    else:
        vertical = "vertically aligned"

    return {
        "target_object": target["label"],
        "reference_object": reference["label"],
        "target_center": [round(tx, 2), round(ty, 2)],
        "reference_center": [round(rx, 2), round(ry, 2)],
        "delta_x_pixels": round(dx, 2),
        "delta_y_pixels": round(dy, 2),
        "center_distance_pixels": round((dx**2 + dy**2)**0.5, 2),
        "horizontal_relationship": horizontal,
        "vertical_relationship": vertical
    }

objects = scene["objects"]
relationships = []

for target, reference in itertools.permutations(objects, 2):
    relationships.append(spatial_relationship(target, reference))

result = {
    "image": scene["image"],
    "coordinate_system": {
        "x_positive": "right",
        "y_positive": "down",
        "distance_units": "pixels, not physical distance"
    },
    "num_objects": len(objects),
    "num_relationships": len(relationships),
    "relationships": relationships
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2)

print("Detected objects:", len(objects))
print("Calculated relationships:", len(relationships))
print("\nSample relationships:")

for item in relationships:
    if (
        item["target_object"] == "dog"
        and item["reference_object"] == "person"
    ) or (
        item["target_object"] == "ball"
        and item["reference_object"] == "dog"
    ):
        print(json.dumps(item, indent=2))

print("\nSaved to:", OUTPUT_PATH)

In [ ]:
import json
import torch

RELATIONSHIPS_PATH = (
    "/content/FastReasonSeg_Results/computed_spatial_relationships.json"
)

with open(RELATIONSHIPS_PATH, "r", encoding="utf-8") as f:
    spatial_data = json.load(f)

def find_relationship(target, reference):
    for item in spatial_data["relationships"]:
        if (
            item["target_object"] == target
            and item["reference_object"] == reference
        ):
            return item
    return None

def answer_spatial_question(target, reference):
    relationship = find_relationship(target, reference)

    if relationship is None:
        return f"No relationship found between {target} and {reference}."

    context = f"""
Image coordinate rules:
X increases to the right.
Y increases downward.

Target object: {target}
Target center: {relationship['target_center']}

Reference object: {reference}
Reference center: {relationship['reference_center']}

Calculated horizontal relationship:
{relationship['horizontal_relationship']}

Calculated vertical relationship:
{relationship['vertical_relationship']}

Calculated center-to-center image distance:
{relationship['center_distance_pixels']} pixels.

Question: Where is the {target} relative to the {reference}?
Answer using the calculated relationships.
"""

    print(f"\nQUESTION: Where is the {target} relative to the {reference}?")

    print("\nBASE MODEL:")
    print(compare_models(base_model, tokenizer_ft, context))

    print("\nFINE-TUNED MODEL:")
    print(compare_models(fine_tuned_model, tokenizer_ft, context))

answer_spatial_question("dog", "person")
answer_spatial_question("ball", "dog")
answer_spatial_question("tree", "dog")
answer_spatial_question("car", "person")

In [ ]:
def spatial_relationship(target, reference):
    dx = target[0] - reference[0]
    dy = target[1] - reference[1]

    horizontal = (
        "left" if dx < 0 else
        "right" if dx > 0 else
        "horizontally aligned"
    )

    vertical = (
        "above" if dy < 0 else
        "below" if dy > 0 else
        "vertically aligned"
    )

    return horizontal, vertical

In [ ]:
import json
import random
import pandas as pd

random.seed(42)

# Generate 50 varied spatial-reasoning questions
objects = ["dog", "person", "car", "ball", "tree", "cat", "chair", "bicycle"]

test_cases = []

for i in range(50):
    target = random.choice(objects)
    reference = random.choice([x for x in objects if x != target])

    # Generate distinct coordinates
    tx, ty = random.randint(50, 900), random.randint(50, 650)
    rx, ry = random.randint(50, 900), random.randint(50, 650)

    # Ensure meaningful separation
    if abs(tx - rx) < 50:
        tx = min(950, tx + 100)
    if abs(ty - ry) < 50:
        ty = min(700, ty + 100)

    dx = tx - rx
    dy = ty - ry

    horizontal = "left" if dx < 0 else "right"
    vertical = "above" if dy < 0 else "below"

    expected = f"{horizontal} and {vertical}"

    prompt = f"""
Image coordinate rules:
X increases to the right.
Y increases downward.

Target object: {target}
Target center: ({tx}, {ty})

Reference object: {reference}
Reference center: ({rx}, {ry})

Question: Where is the {target} relative to the {reference}?
Answer with the horizontal and vertical relationships.
"""

    test_cases.append({
        "id": i + 1,
        "target": target,
        "reference": reference,
        "target_x": tx,
        "target_y": ty,
        "reference_x": rx,
        "reference_y": ry,
        "expected": expected,
        "prompt": prompt
    })

print("Generated test cases:", len(test_cases))
print("Example expected answer:", test_cases[0]["expected"])

In [ ]:
import os
import pandas as pd

path = "/content/FastReasonSeg_Results/spatial_model_comparison.csv"

if os.path.exists(path):
    df = pd.read_csv(path)

    print("Evaluation file found!")
    print("Number of test cases:", len(df))

    print("\nBase model accuracy:",
          round(df["base_correct"].mean() * 100, 2), "%")

    print("Fine-tuned model accuracy:",
          round(df["finetuned_correct"].mean() * 100, 2), "%")

    display(df.head(10))
else:
    print("Evaluation file not found.")
    print("Run the 50-question evaluation cell first.")

In [ ]:
import os
import re
import pandas as pd

OUTPUT_PATH = (
    "/content/FastReasonSeg_Results/spatial_model_comparison.csv"
)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

def extract_relationship(answer):
    text = answer.lower()

    horizontal = None
    vertical = None

    if re.search(r"\bleft\b", text):
        horizontal = "left"
    elif re.search(r"\bright\b", text):
        horizontal = "right"

    if re.search(r"\babove\b|\bupward\b", text):
        vertical = "above"
    elif re.search(r"\bbelow\b|\bdownward\b", text):
        vertical = "below"

    if horizontal and vertical:
        return f"{horizontal} and {vertical}"

    return None


results = []

for i, case in enumerate(test_cases, 1):
    print(f"Evaluating test {i}/{len(test_cases)}", flush=True)

    base_answer = compare_models(
        base_model, tokenizer_ft, case["prompt"]
    )

    ft_answer = compare_models(
        fine_tuned_model, tokenizer_ft, case["prompt"]
    )

    base_prediction = extract_relationship(base_answer)
    ft_prediction = extract_relationship(ft_answer)

    results.append({
        "id": case["id"],
        "target": case["target"],
        "reference": case["reference"],
        "expected": case["expected"],
        "base_prediction": base_prediction,
        "base_correct": base_prediction == case["expected"],
        "finetuned_prediction": ft_prediction,
        "finetuned_correct": ft_prediction == case["expected"],
        "base_answer": base_answer,
        "finetuned_answer": ft_answer
    })

    # Save progress after every test so partial results survive interruptions.
    pd.DataFrame(results).to_csv(OUTPUT_PATH, index=False)

df = pd.DataFrame(results)

print("\n========== EVALUATION RESULTS ==========")
print("Tests completed:", len(df))
print(f"Base model accuracy: {df['base_correct'].mean() * 100:.2f}%")
print(
    "Fine-tuned model accuracy: "
    f"{df['finetuned_correct'].mean() * 100:.2f}%"
)
print(
    "Results saved to:",
    OUTPUT_PATH
)

In [ ]:
print("base_model exists:", "base_model" in globals())
print("fine_tuned_model exists:", "fine_tuned_model" in globals())
print("tokenizer_ft exists:", "tokenizer_ft" in globals())

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen3-0.6B"
ADAPTER_PATH = "/content/FastReasonSeg_Results/qwen3_spatial_lora/final_adapter"

tokenizer_ft = AutoTokenizer.from_pretrained(ADAPTER_PATH)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded successfully!")

In [ ]:
import os

ADAPTER_PATH = "/content/FastReasonSeg_Results/qwen3_spatial_lora/final_adapter"

print("Adapter folder exists:", os.path.isdir(ADAPTER_PATH))

if os.path.isdir(ADAPTER_PATH):
    print("Files:", os.listdir(ADAPTER_PATH))
else:
    print("Parent folder exists:", os.path.isdir(
        "/content/FastReasonSeg_Results/qwen3_spatial_lora"
    ))
    print("Results folder exists:", os.path.isdir(
        "/content/FastReasonSeg_Results"
    ))

In [ ]:
import os

ROOT = "/content/FastReasonSeg_Results"

print("=== Results folder contents ===")
for item in os.listdir(ROOT):
    path = os.path.join(ROOT, item)
    print(("📁 " if os.path.isdir(path) else "📄 ") + item)

print("\n=== Searching for LoRA adapter files ===")
for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in [
        ".cache", "node_modules", "site-packages"
    ]]
    for file in files:
        if file in ["adapter_config.json", "adapter_model.safetensors"]:
            print(os.path.join(root, file))

print("\n=== Searching for project ZIP files ===")
for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in [
        ".cache", "node_modules", "site-packages"
    ]]
    for file in files:
        if "FastReasonSeg" in file and file.endswith(".zip"):
            print(os.path.join(root, file))

In [ ]:
import zipfile
import os

zips = [
    "/content/FastReasonSeg_Final_Project.zip",
    "/content/FastReasonSeg_Prototype_Results.zip"
]

for path in zips:
    print(f"\n{'='*20} {os.path.basename(path)} {'='*20}")

    with zipfile.ZipFile(path, "r") as z:
        matches = [
            name for name in z.namelist()
            if "adapter_config.json" in name
            or "adapter_model.safetensors" in name
            or "spatial_model_comparison.csv" in name
        ]

        if matches:
            for name in matches:
                print(name)
        else:
            print("No LoRA adapter or comparison CSV found.")

In [ ]:
import os
import glob

print("=== Evaluation CSV files ===")
for path in glob.glob("/content/**/*.csv", recursive=True):
    if "spatial_model_comparison" in path:
        print(path)

print("\n=== Saved model adapter files ===")
for path in glob.glob("/content/**/adapter_config.json", recursive=True):
    print(path)

print("\n=== Project ZIP sizes ===")
for path in [
    "/content/FastReasonSeg_Final_Project.zip",
    "/content/FastReasonSeg_Prototype_Results.zip"
]:
    if os.path.exists(path):
        print(os.path.basename(path), round(os.path.getsize(path) / (1024**2), 2), "MB")

In [ ]:
import zipfile

for path in [
    "/content/FastReasonSeg_Final_Project.zip",
    "/content/FastReasonSeg_Prototype_Results.zip"
]:
    print(f"\n{'='*50}\n{path}\n{'='*50}")

    with zipfile.ZipFile(path, "r") as z:
        for name in z.namelist():
            if any(key in name.lower() for key in [
                "dataset", "spatial_sft", "train", "readme",
                "evaluation", "digital_twin", "relationship"
            ]):
                print(name)

In [ ]:
import os

ROOT = "/content/FastReasonSeg_Results"

for name in sorted(os.listdir(ROOT)):
    path = os.path.join(ROOT, name)

    if os.path.isfile(path) and (
        name.endswith(".json") or name.endswith(".jsonl")
    ):
        print(f"FOUND: {path}")

In [ ]:
import json

path = "/content/FastReasonSeg_Results/digital_twin.json"

with open(path, "r") as f:
    data = json.load(f)

print("Data type:", type(data).__name__)

if isinstance(data, dict):
    print("Top-level keys:", list(data.keys()))

print("\nSample data:")
print(json.dumps(data, indent=2)[:5000])

In [ ]:
import json
import os
import math

ROOT = "/content/FastReasonSeg_Results"
INPUT = os.path.join(ROOT, "digital_twin.json")
OUTPUT = os.path.join(ROOT, "spatial_sft_dataset.jsonl")

with open(INPUT, "r") as f:
    scene = json.load(f)

objects = {
    obj["label"]: obj
    for obj in scene["objects"]
}

def center(label):
    return objects[label]["center"]

def relationship(target, reference):
    tx, ty = center(target)
    rx, ry = center(reference)

    dx = tx - rx
    dy = ty - ry

    horizontal = "left" if dx < 0 else "right" if dx > 0 else "aligned"
    vertical = "above" if dy < 0 else "below" if dy > 0 else "aligned"

    distance = math.sqrt(dx**2 + dy**2)

    return horizontal, vertical, dx, dy, distance

examples = []

# Example 1: Relative position
h, v, dx, dy, dist = relationship("dog", "person")
examples.append({
    "instruction": "Where is the dog relative to the person?",
    "input": (
        f"Dog center: {center('dog')}. "
        f"Person center: {center('person')}."
    ),
    "output": (
        f"The dog is {h} and {v} of the person. "
        f"Horizontal offset: {dx:.2f} pixels; "
        f"vertical offset: {dy:.2f} pixels."
    )
})

# Example 2: Center-to-center distance
h, v, dx, dy, dist = relationship("dog", "person")
examples.append({
    "instruction": "What is the distance between the dog and the person?",
    "input": (
        f"Dog center: {center('dog')}. "
        f"Person center: {center('person')}."
    ),
    "output": (
        f"The center-to-center image distance is {dist:.2f} pixels."
    )
})

# Example 3: Horizontal relationship
h, v, dx, dy, dist = relationship("dog", "person")
examples.append({
    "instruction": "Is the dog to the left or right of the person?",
    "input": (
        f"Dog center: {center('dog')}. "
        f"Person center: {center('person')}."
    ),
    "output": f"The dog is to the {h} of the person by {abs(dx):.2f} pixels."
})

# Example 4: Vertical relationship
h, v, dx, dy, dist = relationship("dog", "person")
examples.append({
    "instruction": "Is the dog above or below the person?",
    "input": (
        f"Dog center: {center('dog')}. "
        f"Person center: {center('person')}."
    ),
    "output": f"The dog is {v} the person by {abs(dy):.2f} pixels."
})

# Example 5: Explain the evidence
h, v, dx, dy, dist = relationship("dog", "person")
examples.append({
    "instruction": "Explain the spatial relationship between the dog and person.",
    "input": (
        f"Dog center: {center('dog')}. "
        f"Person center: {center('person')}."
    ),
    "output": (
        f"The dog center is {center('dog')} and the person center is "
        f"{center('person')}. The horizontal difference is {dx:.2f} "
        f"pixels and the vertical difference is {dy:.2f} pixels. "
        f"Therefore, the dog is {h} and {v} of the person."
    )
})

with open(OUTPUT, "w") as f:
    for example in examples:
        f.write(json.dumps(example) + "\n")

print("Dataset recreated:", OUTPUT)
print("Training examples:", len(examples))
print("\nExample:")
print(json.dumps(examples[0], indent=2))

In [ ]:
import os
import torch
import transformers
import peft
import trl

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

print("Dataset exists:", os.path.exists(
    "/content/FastReasonSeg_Results/spatial_sft_dataset.jsonl"
))

In [ ]:
%pip install trl==1.16.0

In [ ]:
import torch
import transformers
import peft
import trl

print("GPU available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

In [ ]:
import os

dataset_path = "/content/FastReasonSeg_Results/spatial_sft_dataset.jsonl"

print("Dataset exists:", os.path.exists(dataset_path))

if os.path.exists(dataset_path):
    with open(dataset_path) as f:
        print("Training examples:", sum(1 for _ in f))

print("GPU:", __import__("torch").cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer

BASE_MODEL = "Qwen/Qwen3-0.6B"
DATASET_PATH = "/content/FastReasonSeg_Results/spatial_sft_dataset.jsonl"
SAVE_PATH = "/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter"

os.makedirs(SAVE_PATH, exist_ok=True)

# Load dataset
examples = []
with open(DATASET_PATH, "r") as f:
    for line in f:
        examples.append(json.loads(line))

# Format examples as instruction-response conversations
rows = []
for item in examples:
    rows.append({
        "text": (
            f"User: {item['instruction']}\n"
            f"Context: {item['input']}\n"
            f"Assistant: {item['output']}"
        )
    })

dataset = Dataset.from_list(rows)

# Load tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Configure LoRA
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

training_args = TrainingArguments(
    output_dir="/content/FastReasonSeg_Results/lora_training",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    logging_steps=1,
    save_strategy="no",
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config
)

trainer.train()

# Save adapter and tokenizer
trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("\nTraining complete.")
print("Adapter saved to:", SAVE_PATH)
print("Saved files:", os.listdir(SAVE_PATH))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-0.6B"
ADAPTER_PATH = "/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto"
)
base_model.eval()

# Load a separate base-model instance for the adapter
ft_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto"
)

fine_tuned_model = PeftModel.from_pretrained(
    ft_base_model,
    ADAPTER_PATH
)
fine_tuned_model.eval()

print("Base model loaded:", base_model is not None)
print("Fine-tuned model loaded:", fine_tuned_model is not None)
print("Adapter loaded:", fine_tuned_model.peft_config.keys())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import re
import random
import pandas as pd
import torch

# -----------------------------
# Configuration
# -----------------------------
OUTPUT_DIR = "/content/drive/MyDrive/FastReasonSeg_Backup/evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(42)

LABELS = [
    "left and above",
    "left and below",
    "right and above",
    "right and below",
]

# -----------------------------
# Generate 50 test cases
# -----------------------------
test_cases = []

while len(test_cases) < 50:
    ax = random.randint(100, 1180)
    ay = random.randint(100, 620)

    # Ensure meaningful horizontal and vertical differences
    bx = random.randint(100, 1180)
    by = random.randint(100, 620)

    if abs(ax - bx) < 100 or abs(ay - by) < 70:
        continue

    horizontal = "right" if ax > bx else "left"
    vertical = "below" if ay > by else "above"
    expected = f"{horizontal} and {vertical}"

    test_cases.append({
        "case": len(test_cases) + 1,
        "object_A_x": ax,
        "object_A_y": ay,
        "object_B_x": bx,
        "object_B_y": by,
        "expected": expected,
    })

# -----------------------------
# Prediction function
# -----------------------------
def predict_relation(model, ax, ay, bx, by):
    prompt = (
        "User: Determine the position of object A relative to object B "
        "using their center coordinates. Image coordinates: x increases "
        "to the right and y increases downward. "
        f"Object A center: [{ax}, {ay}]. "
        f"Object B center: [{bx}, {by}]. "
        "Return exactly one label from these options: "
        "left and above; left and below; right and above; right and below.\n"
        "Assistant:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).lower()

    # Match the predicted relation in the generated answer
    found = re.findall(
        r"\b(left|right)\s+and\s+(above|below)\b",
        generated
    )

    prediction = (
        f"{found[-1][0]} and {found[-1][1]}"
        if found else "unrecognized"
    )

    return prediction, generated.strip()


# -----------------------------
# Evaluate both models
# -----------------------------
results = []

for item in test_cases:
    row = item.copy()

    for model_name, model in [
        ("base", base_model),
        ("fine_tuned", fine_tuned_model),
    ]:
        prediction, raw_answer = predict_relation(
            model,
            item["object_A_x"],
            item["object_A_y"],
            item["object_B_x"],
            item["object_B_y"],
        )

        row[f"{model_name}_prediction"] = prediction
        row[f"{model_name}_raw_answer"] = raw_answer
        row[f"{model_name}_correct"] = (
            prediction == item["expected"]
        )

    results.append(row)

    # Save progress after every case
    pd.DataFrame(results).to_csv(
        os.path.join(OUTPUT_DIR, "spatial_model_comparison.csv"),
        index=False
    )

    print(
        f"Case {item['case']:02d}/50 | "
        f"Expected: {item['expected']:16s} | "
        f"Base: {row['base_prediction']:16s} | "
        f"Fine-tuned: {row['fine_tuned_prediction']}"
    )

# -----------------------------
# Summarize results
# -----------------------------
df = pd.DataFrame(results)

summary = pd.DataFrame([
    {
        "Model": "Base Qwen3-0.6B",
        "Correct": int(df["base_correct"].sum()),
        "Total": len(df),
        "Accuracy (%)": round(df["base_correct"].mean() * 100, 2),
    },
    {
        "Model": "Fine-tuned Qwen3-0.6B + LoRA",
        "Correct": int(df["fine_tuned_correct"].sum()),
        "Total": len(df),
        "Accuracy (%)": round(
            df["fine_tuned_correct"].mean() * 100, 2
        ),
    },
])

summary.to_csv(
    os.path.join(OUTPUT_DIR, "evaluation_summary.csv"),
    index=False
)

print("\n========== EVALUATION SUMMARY ==========")
print(summary.to_string(index=False))

print("\nResults saved to:", OUTPUT_DIR)

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/FastReasonSeg_Backup/evaluation/spatial_model_comparison.csv"
df = pd.read_csv(path)

print("BASE MODEL — predictions")
print(df["base_prediction"].value_counts())

print("\nFINE-TUNED MODEL — predictions")
print(df["fine_tuned_prediction"].value_counts())

print("\nFINE-TUNED ACCURACY BY EXPECTED RELATION")
print(
    df.groupby("expected")["fine_tuned_correct"]
      .agg(["sum", "count", "mean"])
      .assign(accuracy_percent=lambda x: (x["mean"] * 100).round(2))
)

print("\nFIRST 5 RAW FINE-TUNED ANSWERS")
print(df[["expected", "fine_tuned_prediction", "fine_tuned_raw_answer"]].head())

In [ ]:
def spatial_relation(ax, ay, bx, by):
    horizontal = "right" if ax > bx else "left"
    vertical = "below" if ay > by else "above"

    return f"{horizontal} and {vertical}"


# Test cases
tests = [
    (100, 100, 200, 200, "left and above"),
    (100, 300, 200, 200, "left and below"),
    (300, 100, 200, 200, "right and above"),
    (300, 300, 200, 200, "right and below"),
]

correct = 0

for ax, ay, bx, by, expected in tests:
    predicted = spatial_relation(ax, ay, bx, by)
    passed = predicted == expected
    correct += passed

    print(
        f"Expected: {expected:16s} | "
        f"Predicted: {predicted:16s} | "
        f"{'PASS' if passed else 'FAIL'}"
    )

print(f"\nGeometry test accuracy: {correct}/{len(tests)}")

In [ ]:
import json
from itertools import combinations

path = "/content/FastReasonSeg_Results/digital_twin.json"

with open(path, "r") as f:
    data = json.load(f)

objects = data["objects"]

def spatial_relation(ax, ay, bx, by):
    horizontal = "right" if ax > bx else "left"
    vertical = "below" if ay > by else "above"
    return f"{horizontal} and {vertical}"

print("Spatial relationships between detected objects:\n")

for a, b in combinations(objects, 2):
    ax, ay = a["center"]
    bx, by = b["center"]

    relation = spatial_relation(ax, ay, bx, by)

    print(f"{a['label']} is {relation} {b['label']}")

print(f"\nObjects processed: {len(objects)}")
print(f"Relationships calculated: {len(objects) * (len(objects) - 1) // 2}")

In [ ]:
def spatial_relation(ax, ay, bx, by, tolerance=20):
    dx = ax - bx
    dy = ay - by

    if abs(dx) <= tolerance:
        horizontal = "aligned horizontally with"
    elif dx > 0:
        horizontal = "right of"
    else:
        horizontal = "left of"

    if abs(dy) <= tolerance:
        vertical = "aligned vertically with"
    elif dy > 0:
        vertical = "below"
    else:
        vertical = "above"

    return horizontal, vertical

In [ ]:
def spatial_relation(ax, ay, bx, by, tolerance=20):
    dx = ax - bx
    dy = ay - by

    if abs(dx) <= tolerance:
        horizontal = "horizontally aligned with"
    elif dx > 0:
        horizontal = "right of"
    else:
        horizontal = "left of"

    if abs(dy) <= tolerance:
        vertical = "vertically aligned with"
    elif dy > 0:
        vertical = "below"
    else:
        vertical = "above"

    return horizontal, vertical


# Test the function
result = spatial_relation(100, 100, 200, 200)
print("Horizontal relationship:", result[0])
print("Vertical relationship:", result[1])

In [ ]:
import json
from itertools import combinations

path = "/content/FastReasonSeg_Results/digital_twin.json"

with open(path, "r") as f:
    data = json.load(f)

objects = data["objects"]

print("SPATIAL RELATIONSHIP ANALYSIS\n")

for a, b in combinations(objects, 2):
    ax, ay = a["center"]
    bx, by = b["center"]

    horizontal, vertical = spatial_relation(ax, ay, bx, by)

    print(f"{a['label']} vs {b['label']}:")
    print(f"  Horizontal: {horizontal}")
    print(f"  Vertical:   {vertical}")

print(f"\nTotal objects: {len(objects)}")
print(f"Total pairs analyzed: {len(objects) * (len(objects) - 1) // 2}")

In [ ]:
import json

# Load your existing spatial scene
path = "/content/FastReasonSeg_Results/digital_twin.json"

with open(path, "r") as f:
    data = json.load(f)

objects = data["objects"]

# Build structured spatial facts
facts = []

for i, a in enumerate(objects):
    for b in objects[i + 1:]:
        ax, ay = a["center"]
        bx, by = b["center"]

        horizontal, vertical = spatial_relation(ax, ay, bx, by)

        facts.append({
            "object_a": a["label"],
            "object_b": b["label"],
            "horizontal": horizontal,
            "vertical": vertical
        })

# Create a scene description prompt
facts_text = json.dumps(facts, indent=2)

prompt = f"""
Describe the spatial arrangement of objects in this scene.

Use only the following calculated facts. Do not invent relationships.

{facts_text}

Write a concise, readable description of the scene.
"""

# Generate a description using your fine-tuned Qwen3 model
messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text_input,
    return_tensors="pt"
).to(fine_tuned_model.device)

outputs = fine_tuned_model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

generated = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("SCENE DESCRIPTION:\n")
print(generated)

In [ ]:
import json
from itertools import combinations

path = "/content/FastReasonSeg_Results/digital_twin.json"

with open(path, "r") as f:
    data = json.load(f)

objects = data["objects"]
facts = []

for a, b in combinations(objects, 2):
    ax, ay = a["center"]
    bx, by = b["center"]

    horizontal, vertical = spatial_relation(ax, ay, bx, by)

    facts.append(
        f"{a['label']} is {horizontal} {b['label']}; "
        f"vertically, it is {vertical} {b['label']}."
    )

print("VERIFIED GEOMETRIC DESCRIPTION\n")

for i, fact in enumerate(facts, 1):
    print(f"{i}. {fact}")

print(f"\nTotal relationships: {len(facts)}")

In [ ]:
import re

# Ground-truth relationships from your geometry pipeline.
# Format: (object A, object B, horizontal, vertical)
ground_truth = [
    ("dog", "ball", "right", "aligned"),
    ("dog", "person", "right", "below"),
    ("dog", "tree", "left", "below"),
    ("dog", "car", "left", "below"),
    ("ball", "person", "right", "below"),
    ("ball", "tree", "left", "below"),
    ("ball", "car", "left", "below"),
    ("person", "tree", "left", "below"),
    ("person", "car", "left", "below"),
    ("tree", "car", "aligned", "above"),
]

# Previous output generated by Qwen3.
qwen_output = """
The scene features a dog on the right side of the screen,
aligned horizontally with a ball to its left, and a person below it.
A tree to the left of the dog is also below.
A car to the left of the dog is below, and a person to the left
of the tree is below. A ball to the left of the car is below,
and a tree to the car is above.
"""

# Normalize text for simple phrase matching.
text = re.sub(r"\s+", " ", qwen_output.lower())

def horizontal_match(a, b, relation, text):
    if relation == "aligned":
        patterns = [
            rf"{a}.*?aligned horizontally with.*?{b}",
            rf"{b}.*?aligned horizontally with.*?{a}",
        ]
    else:
        patterns = [
            rf"{a}.*?{relation}.*?{b}",
            rf"{b}.*?{('left' if relation == 'right' else 'right')}.*?{a}",
        ]

    return any(re.search(p, text) for p in patterns)

def vertical_match(a, b, relation, text):
    if relation == "aligned":
        return bool(re.search(
            rf"{a}.*?aligned horizontally with.*?{b}", text
        ) or re.search(
            rf"{b}.*?aligned horizontally with.*?{a}", text
        ))

    opposite = "above" if relation == "below" else "below"

    patterns = [
        rf"{a}.*?{relation}.*?{b}",
        rf"{b}.*?{opposite}.*?{a}",
    ]

    return any(re.search(p, text) for p in patterns)

# Evaluate each expected relationship.
results = []

for a, b, horizontal, vertical in ground_truth:
    h_ok = horizontal_match(a, b, horizontal, text)
    v_ok = vertical_match(a, b, vertical, text)

    results.append({
        "Pair": f"{a} - {b}",
        "Horizontal": "PASS" if h_ok else "MISS",
        "Vertical": "PASS" if v_ok else "MISS",
        "Both correct": h_ok and v_ok,
    })

for result in results:
    print(
        f"{result['Pair']:18} | "
        f"H: {result['Horizontal']:4} | "
        f"V: {result['Vertical']:4} | "
        f"{'PASS' if result['Both correct'] else 'CHECK'}"
    )

passed = sum(r["Both correct"] for r in results)

print(f"\nRelationship coverage: {passed}/{len(results)}")
print(f"Coverage percentage: {passed / len(results) * 100:.1f}%")

In [ ]:
import re

qwen_output = """
The scene features a dog on the right side of the screen,
aligned horizontally with a ball to its left, and a person below it.
A tree to the left of the dog is also below.
A car to the left of the dog is below, and a person to the left
of the tree is below. A ball to the left of the car is below,
and a tree to the car is above.
"""

text = re.sub(r"\s+", " ", qwen_output.lower())

# Check whether the text expresses horizontal alignment.
alignment_patterns = [
    r"tree.{0,50}car.{0,50}aligned horizontally",
    r"car.{0,50}tree.{0,50}aligned horizontally",
    r"tree.{0,50}car.{0,50}horizontally aligned",
    r"car.{0,50}tree.{0,50}horizontally aligned",
]

horizontal_aligned = any(
    re.search(pattern, text)
    for pattern in alignment_patterns
)

# Check the tree/car vertical relationship in either direction.
vertical_patterns = [
    r"tree.{0,30}above.{0,30}car",
    r"car.{0,30}below.{0,30}tree",
]

vertical_above = any(
    re.search(pattern, text)
    for pattern in vertical_patterns
)

print("TREE–CAR RELATIONSHIP CHECK")
print("Horizontal alignment:", "PASS" if horizontal_aligned else "MISS")
print("Vertical relationship:", "PASS" if vertical_above else "MISS")

if horizontal_aligned and vertical_above:
    print("Overall: PASS")
else:
    print("Overall: INCOMPLETE DESCRIPTION")

In [ ]:
from google.colab import files
from PIL import Image
import os

uploaded = files.upload()

test_dir = "/content/FastReasonSeg_Test_Images"
os.makedirs(test_dir, exist_ok=True)

valid_extensions = (".jpg", ".jpeg", ".png", ".webp")
image_paths = []

for filename, content in uploaded.items():
    if filename.lower().endswith(valid_extensions):
        path = os.path.join(test_dir, filename)

        with open(path, "wb") as f:
            f.write(content)

        # Verify the image can be opened
        with Image.open(path) as img:
            img.verify()

        image_paths.append(path)

print(f"\nImages uploaded successfully: {len(image_paths)}")

for path in image_paths:
    print("✓", os.path.basename(path))

if len(image_paths) != 5:
    print("\nPlease upload exactly 5 valid images for this test.")

In [ ]:
import os
import torch

print("GPU available:", torch.cuda.is_available())
print("Google Drive mounted:", os.path.exists("/content/drive/MyDrive"))
print("LoRA adapter saved:", os.path.exists(
    "/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/adapter_model.safetensors"
))

GPU available: True
Google Drive mounted: False
LoRA adapter saved: False


In [ ]:
import os
import torch

print("GPU available:", torch.cuda.is_available())
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))
print("LoRA adapter exists:", os.path.exists(
    "/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/adapter_model.safetensors"
))
print("Previous results exist:", os.path.exists(
    "/content/FastReasonSeg_Results"
))

GPU available: True
Drive mounted: False
LoRA adapter exists: False
Previous results exist: False


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

adapter_path = (
    "/content/drive/MyDrive/FastReasonSeg_Backup/"
    "final_adapter/adapter_model.safetensors"
)

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))
print("LoRA adapter exists:", os.path.exists(adapter_path))

if os.path.exists(adapter_path):
    print("SUCCESS: Your trained adapter is available.")
else:
    print("Adapter not found at this path. We need to locate your backup.")

Drive mounted: True
LoRA adapter exists: True
SUCCESS: Your trained adapter is available.


In [ ]:
import os

backup = "/content/drive/MyDrive/FastReasonSeg_Backup"

if os.path.exists(backup):
    print("✅ Backup folder found")
    for root, dirs, files in os.walk(backup):
        for name in files:
            print(os.path.join(root, name))
else:
    print("❌ Backup folder not found:", backup)

✅ Backup folder found
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/README.md
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/adapter_model.safetensors
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/adapter_config.json
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/chat_template.jinja
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/tokenizer_config.json
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/tokenizer.json
/content/drive/MyDrive/FastReasonSeg_Backup/evaluation/spatial_model_comparison.csv
/content/drive/MyDrive/FastReasonSeg_Backup/evaluation/evaluation_summary.csv


In [ ]:
import pandas as pd
import os

folder = "/content/drive/MyDrive/FastReasonSeg_Backup/evaluation"

for filename in ["spatial_model_comparison.csv", "evaluation_summary.csv"]:
    path = os.path.join(folder, filename)
    print(f"\n{'='*50}\n{filename}")
    df = pd.read_csv(path)
    print(df.to_string(index=False))


spatial_model_comparison.csv
 case  object_A_x  object_A_y  object_B_x  object_B_y        expected base_prediction                                                                                                                                        base_raw_answer  base_correct fine_tuned_prediction                                                                                                                                                     fine_tuned_raw_answer  fine_tuned_correct
    1         328         125         663         350  left and above  left and above                 :\nthe position of object a relative to object b is left and above.\nthe answer is: left and above\nthe answer is: left and above\nthe          True       right and below                             the answer is left and above\nthe answer is left and below; the answer is right and above; the answer is right and below.\nthe answer is left               False
    2         547         338        1134   

In [ ]:
import os

drive = "/content/drive/MyDrive"

for root, dirs, files in os.walk(drive):
    dirs[:] = [d for d in dirs if d not in [
        ".ipynb_checkpoints", "node_modules"
    ]]
    for name in files:
        if any(key in name.lower() for key in [
            "fastreasonseg", "segmented_dog", "digital_twin",
            "temporal_stability", "evaluation_report",
            "project_evaluation", "prototype_results"
        ]):
            print(os.path.join(root, name))

/content/drive/MyDrive/Colab Notebooks/FastReasonSeg_Implementation.ipynb
/content/drive/MyDrive/Colab Notebooks/FastReasonSeg_Project.ipynb


In [2]:
import os

adapter = (
    "/content/drive/MyDrive/FastReasonSeg_Backup/"
    "final_adapter/adapter_model.safetensors"
)

print("Adapter exists:", os.path.isfile(adapter))

if os.path.isfile(adapter):
    print(f"Adapter size: {os.path.getsize(adapter) / (1024**2):.2f} MB")
    print("READY: Saved adapter is accessible.")

Adapter exists: False


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

root = "/content/drive/MyDrive"

print("Drive mounted:", os.path.isdir(root))

for folder in ["FastReasonSeg_Backup", "Colab Notebooks"]:
    path = os.path.join(root, folder)
    print(f"\n{folder}:",
          os.listdir(path)[:20] if os.path.isdir(path) else "NOT FOUND")

Mounted at /content/drive
Drive mounted: True

FastReasonSeg_Backup: ['final_adapter', 'evaluation']

Colab Notebooks: ['Untitled (1)', 'Untitled0.ipynb', 'Untitled1.ipynb', 'hug1.ipynb', 'inference.ipynb', 'Untitled2.ipynb', 'Untitled3.ipynb', 'Untitled4.ipynb', 'day16.ipynb', 'Data_Science_with_GenAI_Assignment.ipynb', 'Untitled5.ipynb', 'Untitled6.ipynb', 'Untitled7.ipynb', 'Untitled', 'Untitled8.ipynb', 'Untitled9.ipynb', 'Untitled10.ipynb', 'Untitled11.ipynb', 'FastReasonSeg_Project.ipynb', 'FastReasonSeg_Implementation.ipynb']


In [4]:
import os

root = "/content/drive/MyDrive"
matches = []

for current, dirs, files in os.walk(root):
    # Skip common large or irrelevant folders
    dirs[:] = [d for d in dirs if d not in {
        ".cache", "node_modules", ".git"
    }]

    if "adapter_model.safetensors" in files:
        matches.append(os.path.join(current, "adapter_model.safetensors"))

print("Adapter files found:")
print("\n".join(matches) if matches else "NONE FOUND")

Adapter files found:
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/adapter_model.safetensors


In [5]:
import os
import json

adapter_dir = "/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter"
adapter_path = os.path.join(adapter_dir, "adapter_model.safetensors")
config_path = os.path.join(adapter_dir, "adapter_config.json")

print("Adapter exists:", os.path.isfile(adapter_path))
print("Adapter size:", round(os.path.getsize(adapter_path) / (1024**2), 2), "MB"
      if os.path.isfile(adapter_path) else "Missing")
print("Config exists:", os.path.isfile(config_path))

if os.path.isfile(config_path):
    with open(config_path) as f:
        config = json.load(f)
    print("Base model:", config.get("base_model_name_or_path"))

Adapter exists: True
Adapter size: 8.78 MB
Config exists: True
Base model: Qwen/Qwen3-0.6B


In [7]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Paths
adapter_dir = "/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter"

# Read the original base model from the saved adapter configuration
with open(os.path.join(adapter_dir, "adapter_config.json")) as f:
    config = json.load(f)

base_model_name = config["base_model_name_or_path"]

print("Base model:", base_model_name)
print("GPU available:", torch.cuda.is_available())

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)

# Load the original base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Attach your saved fine-tuned LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    adapter_dir
)

model.eval()

print("\nSUCCESS: Fine-tuned model loaded!")
print("Model device:", model.device)

Base model: Qwen/Qwen3-0.6B
GPU available: True


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [8]:
%pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 14.9 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import torchao
import peft

print("TorchAO:", torchao.__version__)
print("PEFT:", peft.__version__)

TorchAO: 0.18.0
PEFT: 0.21.2


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

adapter_dir = "/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter"

with open(os.path.join(adapter_dir, "adapter_config.json")) as f:
    config = json.load(f)

base_model_name = config["base_model_name_or_path"]

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

print("✅ Fine-tuned model loaded successfully!")
print("GPU available:", torch.cuda.is_available())
print("Base model:", base_model_name)
print("Adapter:", adapter_dir)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

✅ Fine-tuned model loaded successfully!
GPU available: True
Base model: Qwen/Qwen3-0.6B
Adapter: /content/drive/MyDrive/FastReasonSeg_Backup/final_adapter


In [13]:
import torch

test_questions = [
    "A dog is to the left of a person. What is the spatial relationship?",
    "A cat is above a chair. What is the spatial relationship?",
    "A car is to the right of a tree. What is the spatial relationship?",
    "A ball is below a table. What is the spatial relationship?",
    "A person is in front of a car. What is the spatial relationship?"
]

for question in test_questions:
    prompt = f"""Answer the spatial relationship in a short phrase.
Question: {question}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print("Question:", question)
    print("Answer:", answer.strip())
    print("-" * 50)

Question: A dog is to the left of a person. What is the spatial relationship?
Answer: The dog is to the left of the person.
Okay, let's see. The question says a dog is to the left of a person. The answer is just "The dog is to the left
--------------------------------------------------
Question: A cat is above a chair. What is the spatial relationship?
Answer: The cat is above the chair.
Okay, so the question is about spatial relationships. The cat is above a chair. The answer is just a short phrase. Let me think. The key here is
--------------------------------------------------
Question: A car is to the right of a tree. What is the spatial relationship?
Answer: The car is to the right of the tree.
Okay, let's see. The question says a car is to the right of a tree. The answer is just "The car is to the right
--------------------------------------------------
Question: A ball is below a table. What is the spatial relationship?
Answer: The ball is below the table.
Okay, let's see. The q

In [14]:
import torch

question = "What is the spatial relationship? Object A is a dog. Object B is a person. Relationship: left. Return only the relationship label."

messages = [
    {"role": "user", "content": question}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Predicted relationship:", answer.strip())

Predicted relationship: left


In [15]:
import torch

relationships = [
    "left", "right", "above", "below",
    "in front of", "behind", "overlapping"
]

for relation in relationships:
    question = (
        f"Object A is a dog. Object B is a person. "
        f"The given relationship is '{relation}'. "
        f"Return only the relationship label."
    )

    messages = [{"role": "user", "content": question}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print(f"Expected: {relation:12} | Predicted: {answer}")

Expected: left         | Predicted: left
Expected: right        | Predicted: right
Expected: above        | Predicted: relationship
Expected: below        | Predicted: relationship
Expected: in front of  | Predicted: relationship: in front of
Expected: behind       | Predicted: behind
Expected: overlapping  | Predicted: overlapping


In [16]:
import os

root = "/content/drive/MyDrive/FastReasonSeg_Backup"

for current, dirs, files in os.walk(root):
    for filename in files:
        if filename.endswith((".json", ".csv", ".npz", ".npy", ".mp4")):
            path = os.path.join(current, filename)
            print(path)

/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/adapter_config.json
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/tokenizer_config.json
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/tokenizer.json
/content/drive/MyDrive/FastReasonSeg_Backup/evaluation/spatial_model_comparison.csv
/content/drive/MyDrive/FastReasonSeg_Backup/evaluation/evaluation_summary.csv


In [17]:
import os

backup = "/content/drive/MyDrive/FastReasonSeg_Backup"

folders = [
    "recovered_results",
    "recovered_results/images",
    "recovered_results/videos",
    "recovered_results/evaluation"
]

for folder in folders:
    path = os.path.join(backup, folder)
    os.makedirs(path, exist_ok=True)
    print("Ready:", path)

Ready: /content/drive/MyDrive/FastReasonSeg_Backup/recovered_results
Ready: /content/drive/MyDrive/FastReasonSeg_Backup/recovered_results/images
Ready: /content/drive/MyDrive/FastReasonSeg_Backup/recovered_results/videos
Ready: /content/drive/MyDrive/FastReasonSeg_Backup/recovered_results/evaluation


In [18]:
from google.colab import drive
drive.mount('/content/drive')

import os

backup = "/content/drive/MyDrive/FastReasonSeg_Backup"
for folder in ["recovered_results/images",
               "recovered_results/videos",
               "recovered_results/evaluation"]:
    os.makedirs(os.path.join(backup, folder), exist_ok=True)

print("Recovery folders ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Recovery folders ready.


In [19]:
import os

locations = [
    "/content/FastReasonSeg_Results",
    "/content/FastReasonSeg",
    "/content/drive/MyDrive/FastReasonSeg_Backup",
]

extensions = (
    ".png", ".jpg", ".jpeg", ".mp4", ".avi",
    ".npy", ".npz", ".json", ".csv", ".txt"
)

for location in locations:
    print(f"\n{'='*20}\nSearching: {location}")

    if not os.path.exists(location):
        print("Folder not found.")
        continue

    found = 0
    for root, dirs, files in os.walk(location):
        for filename in files:
            if filename.lower().endswith(extensions):
                path = os.path.join(root, filename)
                size_mb = os.path.getsize(path) / (1024 * 1024)
                print(f"{path} ({size_mb:.2f} MB)")
                found += 1

    print(f"Files found: {found}")


Searching: /content/FastReasonSeg_Results
Folder not found.

Searching: /content/FastReasonSeg
Folder not found.

Searching: /content/drive/MyDrive/FastReasonSeg_Backup
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/adapter_config.json (0.00 MB)
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/tokenizer_config.json (0.00 MB)
/content/drive/MyDrive/FastReasonSeg_Backup/final_adapter/tokenizer.json (10.89 MB)
/content/drive/MyDrive/FastReasonSeg_Backup/evaluation/spatial_model_comparison.csv (0.02 MB)
/content/drive/MyDrive/FastReasonSeg_Backup/evaluation/evaluation_summary.csv (0.00 MB)
Files found: 5


In [20]:
import os

video_paths = [
    "/content/01_dog.mp4",
    "/content/drive/MyDrive/01_dog.mp4",
    "/content/drive/MyDrive/FastReasonSeg_Backup/recovered_results/videos/01_dog.mp4"
]

for path in video_paths:
    print(f"{'FOUND' if os.path.isfile(path) else 'MISSING'}: {path}")


MISSING: /content/01_dog.mp4
MISSING: /content/drive/MyDrive/01_dog.mp4
MISSING: /content/drive/MyDrive/FastReasonSeg_Backup/recovered_results/videos/01_dog.mp4


In [21]:
import os

root = "/content/drive/MyDrive"
extensions = (".mp4", ".mov", ".avi", ".mkv", ".webm")

print("Searching Google Drive for video files...\n")

found = []

for current, dirs, files in os.walk(root):
    # Skip common folders that aren't needed for recovery
    dirs[:] = [
        d for d in dirs
        if d not in [".Trash", ".ipynb_checkpoints"]
    ]

    for filename in files:
        if filename.lower().endswith(extensions):
            path = os.path.join(current, filename)
            size_mb = os.path.getsize(path) / (1024 * 1024)
            found.append((path, size_mb))

for path, size_mb in found:
    print(f"{size_mb:.2f} MB | {path}")

print(f"\nTotal video files found: {len(found)}")

Searching Google Drive for video files...


Total video files found: 0


In [24]:
import os
import urllib.request
import cv2

url = "https://dl.fbaipublicfiles.com/segment_anything_2/assets/01_dog.mp4"
video_path = "/content/01_dog.mp4"

try:
    request = urllib.request.Request(
        url, headers={"User-Agent": "Mozilla/5.0"}
    )

    with urllib.request.urlopen(request, timeout=60) as response:
        with open(video_path, "wb") as f:
            f.write(response.read())

    cap = cv2.VideoCapture(video_path)
    valid = cap.isOpened()
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if valid and frames > 0:
        print("SUCCESS: Video recovered")
        print("Size:", round(os.path.getsize(video_path) / 1048576, 2), "MB")
        print("Frames:", frames)
        print("Resolution:", width, "x", height)
    else:
        os.remove(video_path)
        print("Downloaded file is not a valid video.")

except Exception as e:
    print("Download failed:", type(e).__name__, str(e)[:300])

Download failed: HTTPError HTTP Error 403: Forbidden


In [25]:
import os
import urllib.request
import cv2

urls = [
    "https://raw.githubusercontent.com/facebookresearch/sam2/main/demo/data/gallery/01_dog.mp4",
    "https://github.com/facebookresearch/sam2/raw/main/demo/data/gallery/01_dog.mp4",
]

path = "/content/01_dog.mp4"

for url in urls:
    try:
        req = urllib.request.Request(
            url, headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(req, timeout=60) as response:
            data = response.read()

        if len(data) < 100_000:
            print("Invalid or incomplete download.")
            continue

        with open(path, "wb") as f:
            f.write(data)

        cap = cv2.VideoCapture(path)
        valid = cap.isOpened()
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()

        if valid and frames > 0:
            print("SUCCESS: Video recovered")
            print("Size:", round(len(data) / 1048576, 2), "MB")
            print("Frames:", frames)
            break

        os.remove(path)
        print("Downloaded file is not a valid video.")

    except Exception as e:
        print("Source failed:", type(e).__name__, str(e)[:200])

if not os.path.exists(path):
    print("Both sources failed. We will try another recovery method.")

SUCCESS: Video recovered
Size: 1.66 MB
Frames: 289


In [26]:
import os
import shutil
import cv2

source = "/content/01_dog.mp4"
destination = (
    "/content/drive/MyDrive/FastReasonSeg_Backup/"
    "recovered_results/videos/01_dog.mp4"
)

assert os.path.isfile(source), "Video missing. Download it again first."

os.makedirs(os.path.dirname(destination), exist_ok=True)
shutil.copy2(source, destination)

assert os.path.getsize(destination) > 100_000

cap = cv2.VideoCapture(destination)
print("Video opens:", cap.isOpened())
print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("Resolution:",
      int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "x",
      int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
cap.release()

print("\nPermanently saved:", destination)
print("File size:", round(os.path.getsize(destination) / 1024**2, 2), "MB")

Video opens: True
Frames: 289
Resolution: 1280 x 720

Permanently saved: /content/drive/MyDrive/FastReasonSeg_Backup/recovered_results/videos/01_dog.mp4
File size: 1.66 MB


In [27]:
import os
import torch

video_path = (
    "/content/drive/MyDrive/FastReasonSeg_Backup/"
    "recovered_results/videos/01_dog.mp4"
)

assert os.path.isfile(video_path), "Video not found!"
assert torch.cuda.is_available(), "Enable GPU in Runtime settings."

print("GPU:", torch.cuda.get_device_name(0))

from sam2.sam2_video_predictor import SAM2VideoPredictor

if "video_predictor" not in globals():
    print("Loading SAM 2.1 Small...")
    video_predictor = SAM2VideoPredictor.from_pretrained(
        "facebook/sam2.1-hiera-small"
    )

print("SAM 2.1 Small is ready.")

GPU: Tesla T4


ModuleNotFoundError: No module named 'sam2'

In [ ]:
%cd /content

!git clone https://github.com/facebookresearch/sam2.git
!pip install -e /content/sam2

/content
fatal: destination path 'sam2' already exists and is not an empty directory.
Obtaining file:///content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done
  Created wheel for SAM-2: filename=sam_2-1.0-0.editable-cp313-cp313-linux_x86_64.whl size=13851 sha256=5586f8c4d127cecadd421ee891437ac2ad8bf455dacf652ff3c8aa0ff6ec0bdd
  Stored in directory: /tmp/pip-ephem-wheel-cache-lgx9fxmo/wheels/64/e2/28/d1780fb444d6d91bdaddca8d1cc2203826a9783d0f6c5c6e84
Successfully built SAM-2
  Attempting uninstall: SAM-2
    Found existing installation: SAM-2 1.0
    Uninstalling SAM-2-1.0:
      Successfully uninstalled SAM-2-1.0
